In [3]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Aug 24 01:17:08 2025"

In [4]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,681275,36.4,1439377,76.9,1439377,76.9
Vcells,1268403,9.7,8388608,64.0,2017910,15.4


In [5]:
# ================= ENSAMBLE WF 30,19,18,17,14 (sin gsutil) =================
if (!require(data.table)) install.packages("data.table"); library(data.table)

WF_IDS    <- c(30, 19, 18, 17, 14)
WF_LABELS <- sprintf("WF95%02d", WF_IDS)   # WF9530, WF9519, ...

# Bases de búsqueda: 1) cwd, 2) EXP_BASE si la seteás, 3) /content/buckets/*/exp
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]

if (!length(BASES)) stop("No encontré bases de búsqueda. Seteá EXP_BASE o montá /content/buckets/...")

# Para cada WF, tomo el primer prediccion.txt que exista en alguna base
find_pred <- function(wf) {
  for (b in BASES) {
    p <- file.path(b, wf, "prediccion.txt")
    if (file.exists(p)) return(normalizePath(p))
  }
  # también pruebo relativo a cwd: ./WF95xx/prediccion.txt
  p2 <- file.path(wf, "prediccion.txt")
  if (file.exists(p2)) return(normalizePath(p2))
  return(NA_character_)
}

FILES <- vapply(WF_LABELS, find_pred, FUN.VALUE = character(1))

if (anyNA(FILES)) {
  faltan <- WF_LABELS[is.na(FILES)]
  cat("Bases exploradas:\n  ", paste(BASES, collapse = "\n  "), "\n", sep = "")
  stop("Faltan estos prediccion.txt:\n  - ",
       paste(file.path(faltan, "prediccion.txt"), collapse = "\n  - "),
       "\nTip: exportá EXP_BASE='/content/buckets/b1/exp' (o el que uses en el WF) y reintentá.")
}

cat("Voy a ensamblear:\n", paste(sprintf("  %s", FILES), collapse = "\n"), "\n")

# Lector robusto (con o sin headers, y dupes por cliente)
read_pred <- function(path, idx) {
  dt <- fread(path)
  nms <- names(dt)
  if (all(c("numero_de_cliente","prob") %in% nms)) {
    dt <- dt[, .(numero_de_cliente, prob)]
  } else {
    setnames(dt, nms[1:2], c("numero_de_cliente","prob"))
    dt <- dt[, .(numero_de_cliente, prob)]
  }
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm = TRUE)), by = numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}

DTs <- Map(read_pred, FILES, seq_along(FILES))

# Join y promedio simple (si querés pesos, después cambiamos)
ens <- Reduce(function(a,b) merge(a, b, by = "numero_de_cliente", all = TRUE), DTs)
ens[, prob := rowMeans(ens[, -1, with = FALSE], na.rm = TRUE)]
ens[is.na(prob), prob := 0]

# Guardar ensamble
TAG      <- paste(WF_IDS, collapse = "_")
OUT_PRED <- sprintf("ensamble_WF_%s_prediccion.txt", TAG)
fwrite(ens[, .(numero_de_cliente, prob)], OUT_PRED, sep = "\t")
writeLines(FILES, sprintf("ensamble_WF_%s_fuentes.txt", TAG))

cat("OK ✅  Guardado: ", OUT_PRED, "\n", sep = "")



Voy a ensamblear:
   /home/juaniripoll27/buckets/b1/exp/WF9530/prediccion.txt
  /home/juaniripoll27/buckets/b1/exp/WF9519/prediccion.txt
  /home/juaniripoll27/buckets/b1/exp/WF9518/prediccion.txt
  /home/juaniripoll27/buckets/b1/exp/WF9517/prediccion.txt
  /home/juaniripoll27/buckets/b1/exp/WF9514/prediccion.txt 
OK ✅  Guardado: ensamble_WF_30_19_18_17_14_prediccion.txt


In [6]:
# ================= SUBMIT A KAGGLE DESDE ENSAMBLE =================
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

WF_IDS   <- c(30, 19, 18, 17, 14)
TAG      <- paste(WF_IDS, collapse = "_")
OUT_PRED <- sprintf("ensamble_WF_%s_prediccion.txt", TAG)
if (!file.exists(OUT_PRED)) stop("No encuentro ", OUT_PRED, ". Corré antes el bloque A.")

ens <- fread(OUT_PRED)

# Defaults (si no hay PARAM.yml)
COMPETITION <- "data-mining-analista-sr-2025-a"
CORTES      <- seq(10000, 12000, by = 500)
EXP_NAME    <- paste0("ENS_WF", TAG)

# Busco PARAM.yml: 1) cwd, 2) en alguno de los WF usados
param_candidates <- c("PARAM.yml")
# agrego PARAM.yml de cada WF si existe
BASES <- unique(c(getwd(), Sys.getenv("EXP_BASE", unset = NA_character_), Sys.glob("/content/buckets/*/exp")))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
WF_LABELS <- sprintf("WF95%02d", WF_IDS)
for (b in BASES) param_candidates <- c(param_candidates, file.path(b, WF_LABELS, "PARAM.yml"))
param_candidates <- unique(param_candidates)

param_path <- param_candidates[file.exists(param_candidates)][1]
if (length(param_path)) {
  P <- try(read_yaml(param_path), silent = TRUE)
  if (!inherits(P, "try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION <- P$kaggle$competencia
    if (!is.null(P$kaggle$cortes))      CORTES      <- as.integer(P$kaggle$cortes)
    if (!is.null(P$experimento))        EXP_NAME    <- paste0(P$experimento, "_ENS_WF", TAG)
  }
}

# Kaggle CLI disponible
chk <- try(system2("kaggle", "--version", stdout = TRUE, stderr = TRUE), silent = TRUE)
if (inherits(chk, "try-error")) stop("Kaggle CLI no disponible en PATH.")

# Orden y submit
setorder(ens, -prob, numero_de_cliente)
dir.create("kaggle", showWarnings = FALSE)

for (k in CORTES) {
  sub <- ens[, .(numero_de_cliente,
                 Predicted = as.integer(seq_len(.N) <= k))]
  fname <- sprintf("kaggle/KA%s_wf%s_%d.csv", EXP_NAME, TAG, k)
  fwrite(sub, fname)
  msg <- sprintf("Ensamble WF %s | top=%d", TAG, k)
  cat("Subiendo:", fname, "...\n")
  res <- system2("kaggle",
                 c("competitions","submit","-c", COMPETITION, "-f", fname, "-m", shQuote(msg)),
                 stdout = TRUE, stderr = TRUE)
  cat(paste(res, collapse = "\n"), "\n")
}

cat("Listo ✅  Submits para cortes: ", paste(CORTES, collapse = ", "), "\n")


Loading required package: yaml



Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_wf30_19_18_17_14_10000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.47MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_wf30_19_18_17_14_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.43MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_wf30_19_18_17_14_11000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.56MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_wf30_19_18_17_14_11500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.56MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_wf30_19_18_17_14_12000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.73MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Listo ✅  Submits para cortes:  10000, 10500, 11000, 11500, 12000 

In [8]:
# ================= DIAGNÓSTICO + BLENDS (FIX) =================
if (!require(data.table)) install.packages("data.table"); library(data.table)

WF_IDS    <- c(30, 19, 18, 17, 14)
WF_LABELS <- sprintf("WF95%02d", WF_IDS)

# Bases de búsqueda (ajustá EXP_BASE si usás otra):
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),   # ej: /content/buckets/b1/exp
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES) > 0)

find_pred <- function(wf) {
  for (b in BASES) {
    p <- file.path(b, wf, "prediccion.txt")
    if (file.exists(p)) return(normalizePath(p))
  }
  p2 <- file.path(wf, "prediccion.txt")
  if (file.exists(p2)) return(normalizePath(p2))
  NA_character_
}
FILES <- vapply(WF_LABELS, find_pred, FUN.VALUE = character(1))
if (anyNA(FILES)) stop("Faltan:\n  - ", paste(file.path(WF_LABELS[is.na(FILES)], "prediccion.txt"), collapse="\n  - "))

cat("Usando archivos:\n", paste(" •", FILES, collapse="\n"), "\n")

read_pred <- function(path, idx) {
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, FILES, seq_along(FILES))

# Unión e intersección
union_dt <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente",all=TRUE),  DTs)
inter    <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente",all=FALSE), DTs)

pred_cols_union <- grep("^p\\d+$", names(union_dt), value=TRUE)
pred_cols_inter <- grep("^p\\d+$", names(inter),    value=TRUE)

# Diagnóstico rápido
cov_tbl <- data.table(model=pred_cols_union, n_clientes=sapply(DTs, nrow))
cat("\nCobertura por archivo:\n"); print(cov_tbl)
cat("\nIntersección exacta de clientes:", nrow(inter), "\n")

# Matrices de correlación (en intersección)
if (length(pred_cols_inter) >= 2) {
  M <- as.matrix(inter[, ..pred_cols_inter])
  cor_pear  <- suppressWarnings(cor(M, use="pairwise.complete.obs"))
  cor_spear <- suppressWarnings(cor(M, method="spearman", use="pairwise.complete.obs"))
  cat("\nCorrelación PEARSON:\n");  print(round(cor_pear,3))
  cat("\nCorrelación SPEARMAN:\n"); print(round(cor_spear,3))
}

# -------- Blends --------
clip01 <- function(x, eps=1e-6) pmin(pmax(x, eps), 1-eps)
logit  <- function(p) log(p/(1-p))
ilogit <- function(z) 1/(1+exp(-z))

# 1) Mean en unión
union_dt[, ens_mean := rowMeans(.SD, na.rm=TRUE), .SDcols=pred_cols_union]

# 2) Rank-mean en intersección
rank_mean <- function(DT, cols) {
  # rank fraccional en [0,1]
  R <- lapply(cols, function(cn) frank(DT[[cn]], ties.method="average")/nrow(DT))
  rowMeans(as.data.frame(R))
}
inter[, ens_rank := rank_mean(inter, pred_cols_inter)]

# 3) Logit-mean en intersección (corrige descalibración)
inter_clip <- copy(inter)
for (cn in pred_cols_inter) inter_clip[[cn]] <- clip01(inter_clip[[cn]])
Z <- as.matrix(inter_clip[, ..pred_cols_inter])
inter[, ens_logit := ilogit(rowMeans(logit(Z)))]

# 4) Diversity-weighted en intersección (w_i ∝ 1 - corr_media_i)
if (length(pred_cols_inter) >= 2) {
  M <- as.matrix(inter[, ..pred_cols_inter])
  C <- suppressWarnings(cor(M, use="pairwise.complete.obs"))
  avg_corr <- (rowSums(C) - 1) / pmax(1, (ncol(C) - 1))
  w <- pmax(0, 1 - avg_corr)
  w <- w / sum(w)
  inter[, ens_div := as.vector(as.matrix(.SD) %*% w), .SDcols=pred_cols_inter]
  cat("\nPesos diversidad (∝ 1 - corr media):", paste(round(w,3), collapse=", "), "\n")
} else {
  inter[, ens_div := get(pred_cols_inter)]
}

# Guardar blends
TAG <- paste(WF_IDS, collapse="_")
fwrite(union_dt[, .(numero_de_cliente, prob = ens_mean)],
       sprintf("ens_mean_WF_%s_prediccion.txt", TAG), sep="\t")
fwrite(inter[, .(numero_de_cliente, prob = ens_rank)],
       sprintf("ens_rank_WF_%s_prediccion.txt", TAG), sep="\t")
fwrite(inter[, .(numero_de_cliente, prob = ens_logit)],
       sprintf("ens_logit_WF_%s_prediccion.txt", TAG), sep="\t")
fwrite(inter[, .(numero_de_cliente, prob = ens_div)],
       sprintf("ens_div_WF_%s_prediccion.txt", TAG), sep="\t")

cat("\n✅ Generados (elige 1 para enviar):\n",
    " - ens_mean_WF_",TAG,"_prediccion.txt (unión)\n",
    " - ens_rank_WF_",TAG,"_prediccion.txt (intersección)\n",
    " - ens_logit_WF_",TAG,"_prediccion.txt (intersección)\n",
    " - ens_div_WF_",TAG,"_prediccion.txt (intersección)\n", sep="")


Usando archivos:
  • /home/juaniripoll27/buckets/b1/exp/WF9530/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9519/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9518/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9517/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9514/prediccion.txt 

Cobertura por archivo:
    model n_clientes
   <char>      <int>
1:     p1     165093
2:     p2     165093
3:     p3     165093
4:     p4     165093
5:     p5     165093

Intersección exacta de clientes: 165093 

Correlación PEARSON:
      p1    p2    p3    p4    p5
p1 1.000 0.951 0.934 0.948 0.958
p2 0.951 1.000 0.965 0.977 0.984
p3 0.934 0.965 1.000 0.986 0.964
p4 0.948 0.977 0.986 1.000 0.979
p5 0.958 0.984 0.964 0.979 1.000

Correlación SPEARMAN:
      p1    p2    p3    p4    p5
p1 1.000 0.971 0.971 0.972 0.972
p2 0.971 1.000 0.988 0.990 0.987
p3 0.971 0.988 1.000 0.991 0.985
p4 0.972 0.990 0.991 1.000 0.989
p5 0.972 0.987 0.985 0.989 1.000

Pesos diversidad (∝ 1 - co

In [9]:
# ================= SUBMITS PARA LOS 4 BLENDS =================
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

WF_IDS <- c(30,19,18,17,14)
TAG    <- paste(WF_IDS, collapse = "_")

# archivos de predicción (de tu bloque anterior)
PRED_FILES <- c(
  sprintf("ens_mean_WF_%s_prediccion.txt",  TAG),
  sprintf("ens_rank_WF_%s_prediccion.txt",  TAG),
  sprintf("ens_logit_WF_%s_prediccion.txt", TAG),
  sprintf("ens_div_WF_%s_prediccion.txt",   TAG)
)
BLEND_NAMES <- c("mean","rank","logit","div")

# Defaults (si no hay PARAM.yml)
COMPETITION    <- "data-mining-analista-sr-2025-a"
CORTES         <- seq(10000, 12000, by = 500)
EXP_NAME_BASE  <- paste0("KA9530_ENS_WF", TAG)  # mismo prefijo que venías usando

# Si existe PARAM.yml, tomo competencia/cortes/experimento
if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent = TRUE)
  if (!inherits(P, "try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION   <- P$kaggle$competencia
    if (!is.null(P$kaggle$cortes))      CORTES        <- as.integer(P$kaggle$cortes)
    if (!is.null(P$experimento))        EXP_NAME_BASE <- paste0(P$experimento, "_ENS_WF", TAG)
  }
}

# Kaggle CLI disponible
chk <- try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE)
if (inherits(chk, "try-error")) stop("Kaggle CLI no disponible en PATH.")

dir.create("kaggle", showWarnings = FALSE)

for (i in seq_along(PRED_FILES)) {
  pred_file <- PRED_FILES[i]
  blend     <- BLEND_NAMES[i]
  if (!file.exists(pred_file)) {
    warning("No existe ", pred_file, " — salteo este blend."); next
  }
  ens <- fread(pred_file)
  if (!all(c("numero_de_cliente","prob") %in% names(ens))) {
    setnames(ens, names(ens)[1:2], c("numero_de_cliente","prob"))
  }
  setorder(ens, -prob, numero_de_cliente)

  for (k in CORTES) {
    sub <- ens[, .(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= k))]
    fname <- sprintf("kaggle/%s_%s_wf%s_%d.csv", EXP_NAME_BASE, blend, TAG, k)
    fwrite(sub, fname)
    msg <- sprintf("Ensamble WF %s | blend=%s | top=%d", TAG, blend, k)
    cat("Subiendo:", fname, "...\n")
    res <- system2("kaggle",
                   c("competitions","submit","-c", COMPETITION, "-f", fname, "-m", shQuote(msg)),
                   stdout = TRUE, stderr = TRUE)
    cat(paste(res, collapse = "\n"), "\n")
  }
}

cat("✅ Listo. Subidos todos los blends para cortes: ", paste(CORTES, collapse=", "), "\n")


Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_mean_wf30_19_18_17_14_10000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.66MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_mean_wf30_19_18_17_14_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.69MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_mean_wf30_19_18_17_14_11000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.51MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_mean_wf30_19_18_17_14_11500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.49MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_mean_wf30_19_18_17_14_12000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.51MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_1

Warning message in system2("kaggle", c("competitions", "submit", "-c", COMPETITION, :
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA9530_ENS_WF30_19_18_17_14_logit_wf30_19_18_17_14_12000.csv -m 'Ensamble WF 30_19_18_17_14 | blend=logit | top=12000' 2>&1' had status 1”


100%|██████████| 1.79M/1.79M [00:00<00:00, 3.59MB/s]
429 Client Error: Too Many Requests for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_div_wf30_19_18_17_14_10000.csv ...


Warning message in system2("kaggle", c("competitions", "submit", "-c", COMPETITION, :
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA9530_ENS_WF30_19_18_17_14_div_wf30_19_18_17_14_10000.csv -m 'Ensamble WF 30_19_18_17_14 | blend=div | top=10000' 2>&1' had status 1”


100%|██████████| 1.79M/1.79M [00:00<00:00, 3.52MB/s]
429 Client Error: Too Many Requests for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_div_wf30_19_18_17_14_10500.csv ...


...

100%|██████████| 1.79M/1.79M [00:00<00:00, 3.64MB/s]
429 Client Error: Too Many Requests for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_div_wf30_19_18_17_14_11500.csv ...


Warning message in system2("kaggle", c("competitions", "submit", "-c", COMPETITION, :
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA9530_ENS_WF30_19_18_17_14_div_wf30_19_18_17_14_11500.csv -m 'Ensamble WF 30_19_18_17_14 | blend=div | top=11500' 2>&1' had status 1”


100%|██████████| 1.79M/1.79M [00:00<00:00, 3.63MB/s]
429 Client Error: Too Many Requests for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Subiendo: kaggle/KA9530_ENS_WF30_19_18_17_14_div_wf30_19_18_17_14_12000.csv ...


Warning message in system2("kaggle", c("competitions", "submit", "-c", COMPETITION, :
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA9530_ENS_WF30_19_18_17_14_div_wf30_19_18_17_14_12000.csv -m 'Ensamble WF 30_19_18_17_14 | blend=div | top=12000' 2>&1' had status 1”


100%|██████████| 1.79M/1.79M [00:00<00:00, 3.58MB/s]
429 Client Error: Too Many Requests for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
✅ Listo. Subidos todos los blends para cortes:  10000, 10500, 11000, 11500, 12000 


In [10]:
# ===== ENSAMBLE GLOBAL: todos los prediccion.txt de WFxxxx (logit + pesos de diversidad) =====
if (!require(data.table)) install.packages("data.table"); library(data.table)

# Bases de búsqueda (ajustá EXP_BASE si hace falta)
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),                     # ej: "/home/juaniripoll27/buckets/b1/exp"
  Sys.glob("/home/juaniripoll27/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES) > 0)

# Recolecto todos los prediccion.txt bajo carpetas WFxxxx
paths <- unique(unlist(lapply(BASES, function(b)
  list.files(b, pattern="^prediccion\\.(txt|csv)$", recursive=TRUE, full.names=TRUE))))
dirs  <- basename(dirname(paths))
keep  <- grepl("^WF\\d+$|^WF95\\d{2}$", dirs)      # WF9530, WF1234, etc.
paths <- paths[keep]

if (!length(paths)) stop("No encontré prediccion.txt dentro de WFxxxx en las bases:\n  ", paste(BASES, collapse="\n  "))

cat("Detectados ", length(paths), " archivos.\n", sep="")

# Leo/normalizo y renombro como p1, p2, ...
read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  # si hubiera duplicados por cliente, promedio interno
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, paths, seq_along(paths))

# Intersección estricta de clientes (para evitar NAs en logit)
inter <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=FALSE), DTs)
pred_cols <- grep("^p\\d+$", names(inter), value=TRUE)

cat("Clientes en intersección: ", nrow(inter), " | modelos: ", length(pred_cols), "\n", sep="")
stopifnot(nrow(inter) > 0, length(pred_cols) >= 2)

# --- Pesos por diversidad: w_i ∝ 1 - corr_media_i (Pearson sobre probs)
M <- as.matrix(inter[, ..pred_cols])
C <- suppressWarnings(cor(M, use="pairwise.complete.obs"))
avg_corr <- (rowSums(C) - 1) / pmax(1, (ncol(C)-1))
w <- pmax(0, 1 - avg_corr)              # diversidad
# Evito que modelos idénticos dominen: piso mínimo chiquito
w <- if (sum(w) == 0) rep(1/length(w), length(w)) else w/sum(w)
cat("Modelos usados y pesos (primeros 10):\n")
print(head(data.table(model=pred_cols, weight=round(w,4)), 10))

# --- Logit-mean ponderado
clip01 <- function(x, eps=1e-6) pmin(pmax(x, eps), 1-eps)
logit  <- function(p) log(p/(1-p))
ilogit <- function(z) 1/(1+exp(-z))

for (cn in pred_cols) inter[[cn]] <- clip01(inter[[cn]])
Z <- as.matrix(inter[, ..pred_cols])
ens_prob <- ilogit(as.vector(Z %*% w))   # combinación logit ponderada

OUT_PRED_ALL <- "ens_all_logit_div_prediccion.txt"
fwrite(inter[, .(numero_de_cliente, prob = ens_prob)], OUT_PRED_ALL, sep="\t")
writeLines(paths, "ens_all_fuentes.txt")

cat("✅ Guardado: ", OUT_PRED_ALL, " (", length(pred_cols), " modelos)\n", sep="")


Detectados 16 archivos.
Clientes en intersección: 33094 | modelos: 16
Modelos usados y pesos (primeros 10):
     model weight
    <char>  <num>
 1:     p1 0.1264
 2:     p2 0.0593
 3:     p3 0.0613
 4:     p4 0.0379
 5:     p5 0.0498
 6:     p6 0.0606
 7:     p7 0.0445
 8:     p8 0.0603
 9:     p9 0.1264
10:    p10 0.0593
✅ Guardado: ens_all_logit_div_prediccion.txt (16 modelos)


In [13]:
# ===== SUBMIT a Kaggle usando el ensamble global (K=10500) =====
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

PRED_FILE <- "ens_all_logit_div_prediccion.txt"
if (!file.exists(PRED_FILE)) stop("No existe ", PRED_FILE, ". Corré el bloque A primero.")

ens <- fread(PRED_FILE)
if (!all(c("numero_de_cliente","prob") %in% names(ens)))
  setnames(ens, names(ens)[1:2], c("numero_de_cliente","prob"))

# Competencia / nombre
COMPETITION   <- "data-mining-analista-sr-2025-a"
EXP_NAME_BASE <- "KA_ALL_ENS"

if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent=TRUE)
  if (!inherits(P, "try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION   <- P$kaggle$competencia
    if (!is.null(P$experimento))        EXP_NAME_BASE <- paste0(P$experimento, "_ALL_ENS")
  }
}

# Kaggle CLI
stopifnot(!inherits(try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE),"try-error"))

K <- 11500L
setorder(ens, -prob, numero_de_cliente)
sub <- ens[, .(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= K))]

dir.create("kaggle", showWarnings = FALSE)
fname <- sprintf("kaggle/%s_logit_div_top_%d.csv", EXP_NAME_BASE, K)
fwrite(sub, fname)

msg <- sprintf("ALL models | logit-mean weighted by diversity | top=%d", K)
cat("Subiendo:", fname, "...\n")
res <- system2("kaggle",
               c("competitions","submit","-c", COMPETITION, "-f", fname, "-m", shQuote(msg)),
               stdout=TRUE, stderr=TRUE)
cat(paste(res, collapse="\n"), "\n")

    

Subiendo: kaggle/KA_ALL_ENS_logit_div_top_11500.csv ...
100%|██████████| 368k/368k [00:00<00:00, 875kB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 


In [14]:
# ===== ENSAMBLE GLOBAL COMPLETO =====
if (!require(data.table)) install.packages("data.table"); library(data.table)

# Bases donde buscar WFxxxx/prediccion.txt
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),                     # ej: "/home/juaniripoll27/buckets/b1/exp"
  Sys.glob("/home/juaniripoll27/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES) > 0)

# 1) Recolecto TODOS los prediccion.txt dentro de carpetas WFxxxx
paths <- unique(unlist(lapply(BASES, function(b)
  list.files(b, pattern="^prediccion\\.(txt|csv)$", recursive=TRUE, full.names=TRUE))))
dirs  <- basename(dirname(paths))
keep  <- grepl("^WF\\d+$|^WF95\\d{2}$", dirs)
paths <- paths[keep]
if (!length(paths)) stop("No encontré prediccion.txt dentro de WFxxxx en:\n  ", paste(BASES, collapse="\n  "))

cat("Detectados ", length(paths), " archivos de predicción.\n", sep="")

# 2) Lectura normalizada
read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, paths, seq_along(paths))

# 3) Para estimar correlaciones → intersección estricta
inter <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=FALSE), DTs)
pred_cols_inter <- grep("^p\\d+$", names(inter), value=TRUE)
stopifnot(nrow(inter) > 0, length(pred_cols_inter) >= 2)

# Pesos por diversidad: w ∝ (1 - corr media)
M <- as.matrix(inter[, ..pred_cols_inter])
C <- suppressWarnings(cor(M, use="pairwise.complete.obs"))
avg_corr <- (rowSums(C) - 1) / pmax(1, (ncol(C)-1))
w <- pmax(0, 1 - avg_corr)
w <- if (sum(w)==0) rep(1/length(w), length(w)) else w/sum(w)

cat("Modelos y pesos (primeros 10):\n")
print(head(data.table(model=pred_cols_inter, weight=round(w,4)), 10))

# 4) BASE DE REFERENCIA COMPLETA (garantiza mismo universo que Kaggle)
#    usamos, en este orden: tu mejor ensamble de 5 modelos, o un WF que haya funcionado
ref_candidates <- c(
  "ens_logit_WF_30_19_18_17_14_prediccion.txt",
  file.path(BASES[1], "WF9530", "prediccion.txt"),
  file.path(BASES[1], "WF9519", "prediccion.txt")
)
ref_path <- ref_candidates[file.exists(ref_candidates)][1]
if (length(ref_path)==0) stop("No encontré base de referencia completa. Apuntá ref_candidates a un pred que haya pasado en Kaggle.")
REF <- fread(ref_path)[, .(numero_de_cliente)]
cat("Referencia tomada de:", ref_path, " | n=", nrow(REF), "\n")

# 5) Unión de todos los modelos sobre la referencia
union_all <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=TRUE), DTs)
ALL <- merge(REF, union_all, by="numero_de_cliente", all.x=TRUE)
pred_cols_all <- grep("^p\\d+$", names(ALL), value=TRUE)

# 6) Logit-mean ponderado con re-normalización por fila (maneja NAs)
clip01 <- function(x, eps=1e-6) pmin(pmax(x, eps), 1-eps)
logit  <- function(p) log(p/(1-p))
ilogit <- function(z) 1/(1+exp(-z))

# Alineo vector de pesos w (definido en intersección) al orden de columnas en ALL.
# Para los modelos que no estaban en la intersección (raro), les doy peso mínimo.
map_w <- rep(min(w)/10, length(pred_cols_all))  # piso chico
names(map_w) <- pred_cols_all
names(w) <- pred_cols_inter
map_w[names(w)] <- w

P <- as.matrix(ALL[, ..pred_cols_all])           # probabilidades
P <- pmin(pmax(P, 1e-6), 1-1e-6)                 # clip
W <- matrix(map_w, nrow=nrow(P), ncol=ncol(P), byrow=TRUE)
W[is.na(P)] <- 0                                  # si falta un modelo en esa fila, no pesa
Z <- logit(P); Z[!is.finite(Z)] <- 0

wsum <- rowSums(W)
W[wsum>0, ] <- W[wsum>0, ] / wsum[wsum>0]        # re-normalizo pesos por fila
ens_prob <- ilogit(rowSums(Z * W))

OUT_ALL <- "ens_all_logit_div_FULL_prediccion.txt"
fwrite(data.table(numero_de_cliente = ALL$numero_de_cliente, prob = ens_prob),
       OUT_ALL, sep="\t")
cat("✅ Guardado:", OUT_ALL, " | filas:", nrow(ALL), "\n")


Detectados 16 archivos de predicción.
Modelos y pesos (primeros 10):
     model weight
    <char>  <num>
 1:     p1 0.1264
 2:     p2 0.0593
 3:     p3 0.0613
 4:     p4 0.0379
 5:     p5 0.0498
 6:     p6 0.0606
 7:     p7 0.0445
 8:     p8 0.0603
 9:     p9 0.1264
10:    p10 0.0593
Referencia tomada de: ens_logit_WF_30_19_18_17_14_prediccion.txt  | n= 165093 
✅ Guardado: ens_all_logit_div_FULL_prediccion.txt  | filas: 165093 


In [17]:
# ===== SUBMIT K=10500 con el ensamble FULL =====
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

PRED_FILE <- "ens_all_logit_div_FULL_prediccion.txt"
if (!file.exists(PRED_FILE)) stop("No existe ", PRED_FILE, ". Corré el ensamble FULL primero.")

ens <- fread(PRED_FILE)
if (!all(c("numero_de_cliente","prob") %in% names(ens)))
  setnames(ens, names(ens)[1:2], c("numero_de_cliente","prob"))

COMPETITION   <- "data-mining-analista-sr-2025-a"
EXP_NAME_BASE <- "KA_ALL_ENS_FULL"

if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent=TRUE)
  if (!inherits(P, "try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION   <- P$kaggle$competencia
    if (!is.null(P$experimento))        EXP_NAME_BASE <- paste0(P$experimento, "_ALL_ENS_FULL")
  }
}

stopifnot(!inherits(try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE),"try-error"))

K <- 11500L
setorder(ens, -prob, numero_de_cliente)
sub <- ens[, .(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= K))]

dir.create("kaggle", showWarnings = FALSE)
fname <- sprintf("kaggle/%s_logit_div_top_%d.csv", EXP_NAME_BASE, K)
fwrite(sub, fname)
msg <- sprintf("ALL models | logit-mean weighted by diversity | FULL | top=%d", K)

cat("Subiendo:", fname, "...\n")
res <- system2("kaggle", c("competitions","submit","-c", COMPETITION, "-f", fname, "-m", shQuote(msg)),
               stdout=TRUE, stderr=TRUE)
cat(paste(res, collapse="\n"), "\n")


Subiendo: kaggle/KA_ALL_ENS_FULL_logit_div_top_11500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.64MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 


In [18]:
# ============== BLENDS FULL desde TODOS los WF ==============
if (!require(data.table)) install.packages("data.table"); library(data.table)

# Bases típicas (ajustá EXP_BASE si querés fijarla)
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),           # ej: "/home/juaniripoll27/buckets/b1/exp"
  Sys.glob("/home/juaniripoll27/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES) > 0)

# Recolecto predicciones bajo WFxxxx
paths <- unique(unlist(lapply(BASES, function(b)
  list.files(b, pattern="^prediccion\\.(txt|csv)$", recursive=TRUE, full.names=TRUE))))
dirs  <- basename(dirname(paths))
paths <- paths[grepl("^WF\\d+$|^WF95\\d{2}$", dirs)]
if (!length(paths)) stop("No encontré prediccion.txt en WFxxxx dentro de:\n  ", paste(BASES, collapse="\n  "))

cat("Detectados ", length(paths), " archivos de predicción.\n", sep="")

# Lector normalizado
read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, paths, seq_along(paths))

# ----- Pesos de diversidad (calculados en intersección) -----
inter <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=FALSE), DTs)
pred_cols_inter <- grep("^p\\d+$", names(inter), value=TRUE)
stopifnot(nrow(inter) > 0, length(pred_cols_inter) >= 2)

M <- as.matrix(inter[, ..pred_cols_inter])
C <- suppressWarnings(cor(M, use="pairwise.complete.obs"))
avg_corr <- (rowSums(C) - 1) / pmax(1, (ncol(C)-1))
w_div <- pmax(0, 1 - avg_corr)
w_div <- if (sum(w_div)==0) rep(1/length(w_div), length(w_div)) else w_div/sum(w_div)
names(w_div) <- pred_cols_inter
cat("Modelos en intersección:", length(w_div), " | ejemplo de pesos:\n")
print(head(round(w_div,3)))

# ----- Base de referencia completa -----
ref_candidates <- c(
  "ens_logit_WF_30_19_18_17_14_prediccion.txt", # tu mejor de 5
  paths[1]                                      # cualquier WF
)
ref_path <- ref_candidates[file.exists(ref_candidates)][1]
if (!length(ref_path)) stop("No encontré referencia completa; ajustá 'ref_candidates'.")
REF <- fread(ref_path)[, .(numero_de_cliente)]
cat("Referencia:", ref_path, " | n=", nrow(REF), "\n")

# ----- Unión sobre referencia (garantiza FULL) -----
union_all <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=TRUE), DTs)
ALL <- merge(REF, union_all, by="numero_de_cliente", all.x=TRUE)
pred_cols_all <- grep("^p\\d+$", names(ALL), value=TRUE)

# utilidades
clip01 <- function(x, eps=1e-6) pmin(pmax(x, eps), 1-eps)
logit  <- function(p) log(p/(1-p))
ilogit <- function(z) 1/(1+exp(-z))

P <- as.matrix(ALL[, ..pred_cols_all])                 # matrix de probs (con NAs)
Pclip <- pmin(pmax(P, 1e-6), 1-1e-6)                   # para transforms

# map de pesos de diversidad para todas las columnas (piso chico a las que no están en inter)
map_w <- rep(min(w_div)/10, length(pred_cols_all)); names(map_w) <- pred_cols_all
map_w[names(w_div)] <- w_div
W <- matrix(map_w, nrow=nrow(P), ncol=ncol(P), byrow=TRUE)
W[is.na(P)] <- 0                                       # si falta el modelo en esa fila, no pesa
wsum <- rowSums(W); W[wsum>0,] <- W[wsum>0,] / wsum[wsum>0]   # renormaliza por fila

# ----- BLENDS FULL -----
# 1) mean_full
ens_mean_full <- rowMeans(P, na.rm=TRUE)

# 2) median_full
ens_median_full <- apply(P, 1, function(v){ v <- v[is.finite(v)]; if(length(v)) median(v, na.rm=TRUE) else NA_real_ })

# 3) tmean_full (descarto min y max si hay >=3)
ens_tmean_full <- apply(P, 1, function(v){
  v <- v[is.finite(v)]
  if (length(v) >= 3) { v <- sort(v); mean(v[-c(1,length(v))]) }
  else if (length(v) > 0) mean(v) else NA_real_
})

# 4) rank_full (promedio de percentiles por columna)
percentile_col <- function(v) {
  r <- rep(NA_real_, length(v)); ok <- is.finite(v)
  r[ok] <- frank(v[ok], ties.method="average")/sum(ok)
  r
}
R <- do.call(cbind, lapply(seq_along(pred_cols_all), function(j) percentile_col(ALL[[pred_cols_all[j]]])))
ens_rank_full <- rowMeans(R, na.rm=TRUE)

# 5) logit_full (promedio simple en logit)
Z <- logit(Pclip)
ens_logit_full <- ilogit(rowMeans(Z, na.rm=TRUE))

# 6) logit_div_full (logit con pesos de diversidad + renormalización por fila)
ens_logit_div_full <- ilogit(rowSums(Z * W))

# ----- Guardar -----
out <- function(name, vec){
  fname <- paste0("ens_", name, "_FULL_prediccion.txt")
  fwrite(data.table(numero_de_cliente = ALL$numero_de_cliente, prob = vec), fname, sep="\t")
  cat("✓", fname, "\n")
}
out("mean",   ens_mean_full)
out("median", ens_median_full)
out("tmean",  ens_tmean_full)
out("rank",   ens_rank_full)
out("logit",  ens_logit_full)
out("logit_div", ens_logit_div_full)


Detectados 16 archivos de predicción.
Modelos en intersección: 16  | ejemplo de pesos:
   p1    p2    p3    p4    p5    p6 
0.126 0.059 0.061 0.038 0.050 0.061 
Referencia: ens_logit_WF_30_19_18_17_14_prediccion.txt  | n= 165093 
✓ ens_mean_FULL_prediccion.txt 
✓ ens_median_FULL_prediccion.txt 
✓ ens_tmean_FULL_prediccion.txt 
✓ ens_rank_FULL_prediccion.txt 
✓ ens_logit_FULL_prediccion.txt 
✓ ens_logit_div_FULL_prediccion.txt 


In [19]:
# ============== SUBMIT de blends FULL (K=10500/11000/11500) ==============
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

BLENDS <- c("mean","median","tmean","rank","logit","logit_div")  # podés comentar algunos
K_SET  <- c(10500L, 11000L, 11500L)

COMPETITION   <- "data-mining-analista-sr-2025-a"
EXP_NAME_BASE <- "KA_ALL_FULL"

if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent=TRUE)
  if (!inherits(P, "try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION   <- P$kaggle$competencia
    if (!is.null(P$experimento))        EXP_NAME_BASE <- paste0(P$experimento, "_ALL_FULL")
  }
}

stopifnot(!inherits(try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE),"try-error"))
dir.create("kaggle", showWarnings = FALSE)

for (b in BLENDS) {
  pred <- sprintf("ens_%s_FULL_prediccion.txt", b)
  if (!file.exists(pred)) { warning("Falta ", pred); next }
  ens <- fread(pred)
  if (!all(c("numero_de_cliente","prob") %in% names(ens)))
    setnames(ens, names(ens)[1:2], c("numero_de_cliente","prob"))
  setorder(ens, -prob, numero_de_cliente)

  for (k in K_SET) {
    fname <- sprintf("kaggle/%s_%s_top_%d.csv", EXP_NAME_BASE, b, k)
    sub   <- ens[, .(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= k))]
    fwrite(sub, fname)
    msg <- sprintf("ALL FULL | blend=%s | top=%d", b, k)
    cat("Subiendo:", fname, "...\n")
    res <- system2("kaggle",
                   c("competitions","submit","-c", COMPETITION, "-f", fname, "-m", shQuote(msg)),
                   stdout=TRUE, stderr=TRUE)
    cat(paste(res, collapse="\n"), "\n")
    Sys.sleep(8)   # pequeña pausa para evitar 429
  }
}
cat("Hecho. Revisá el LB por blend y K.\n")


Subiendo: kaggle/KA_ALL_FULL_mean_top_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.44MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_ALL_FULL_mean_top_11000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.71MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_ALL_FULL_mean_top_11500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.63MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_ALL_FULL_median_top_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.52MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_ALL_FULL_median_top_11000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.58MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_ALL_FULL_median_top_11500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.64MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle

In [20]:
# ========== SUBSET INTELIGENTE + BLENDS FULL ==========
if (!require(data.table)) install.packages("data.table"); library(data.table)

# Bases donde buscar WFxxxx
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),           # ej: "/home/juaniripoll27/buckets/b1/exp"
  Sys.glob("/home/juaniripoll27/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES) > 0)

# 1) Recolectar todos los prediccion.txt bajo WFxxxx
paths <- unique(unlist(lapply(BASES, function(b)
  list.files(b, pattern="^prediccion\\.(txt|csv)$", recursive=TRUE, full.names=TRUE))))
dirs  <- basename(dirname(paths))
paths <- paths[grepl("^WF\\d+$|^WF95\\d{2}$", dirs)]
if (!length(paths)) stop("No encontré prediccion.txt en WFxxxx dentro de:\n  ", paste(BASES, collapse="\n  "))

cat("Encontrados", length(paths), "archivos.\n")

# lector normalizado
read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, paths, seq_along(paths))

# 2) Estimar diversidad y acuerdo en la INTERSECCIÓN (sin NAs)
inter <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=FALSE), DTs)
pred_cols <- grep("^p\\d+$", names(inter), value=TRUE)
stopifnot(nrow(inter) > 0, length(pred_cols) >= 2)

M <- as.matrix(inter[, ..pred_cols])
C <- suppressWarnings(cor(M, use="pairwise.complete.obs"))
avg_corr <- (rowSums(C) - 1) / pmax(1, (ncol(C)-1))
diversity <- 1 - avg_corr                  # mayor = más distinto

# Consenso por rank-mean y acuerdo de top-K
K_CONS <- min(11000L, nrow(inter))         # alrededor del mejor K
rank_mean <- function(mat) {
  R <- apply(mat, 2, function(v) rank(v, ties.method="average")/length(v))
  rowMeans(R)
}
cons <- rank_mean(M)
top_cons <- order(cons, decreasing = TRUE)[1:K_CONS]   # índices consenso

overlap_frac <- sapply(seq_along(pred_cols), function(j){
  top_j <- order(M[,j], decreasing = TRUE)[1:K_CONS]
  length(intersect(top_j, top_cons)) / K_CONS
})

# 3) Scoring y selección del subconjunto
score <- 0.55*diversity + 0.45*overlap_frac
ord   <- order(score, decreasing = TRUE)
n_tot <- length(pred_cols)
N_SEL <- min(max(8, round(0.35*n_tot)), 30)   # ~35% (cap en 30); ajustá si querés
sel_cols <- pred_cols[ord[1:N_SEL]]

cat("Seleccionados", length(sel_cols), "modelos de", n_tot, "\n")
print(head(data.table(model=sel_cols, score=round(score[match(sel_cols, pred_cols)],3)), 10))

# 4) Ensambles FULL: usar referencia completa para cubrir todo el universo
ref_candidates <- c(
  "ens_logit_WF_30_19_18_17_14_prediccion.txt",  # tu ensamble que ya pasó
  paths[1]                                       # o el primer WF
)
ref_path <- ref_candidates[file.exists(ref_candidates)][1]
if (!length(ref_path)) stop("No hay referencia completa. Ajustá 'ref_candidates'.")
REF <- fread(ref_path)[, .(numero_de_cliente)]
cat("Referencia:", ref_path, " | n=", nrow(REF), "\n")

# Unión de TODOS los modelos sobre la referencia
union_all <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=TRUE), DTs)
ALL <- merge(REF, union_all, by="numero_de_cliente", all.x=TRUE)

subset_cols <- sel_cols                               # columnas del subset
P_sub  <- as.matrix(ALL[, ..subset_cols])
Pclip  <- pmin(pmax(P_sub, 1e-6), 1-1e-6)

# pesos de diversidad restringidos al subset
w_div <- pmax(0, diversity[match(subset_cols, pred_cols)])
w_div <- if (sum(w_div)==0) rep(1/length(subset_cols), length(subset_cols)) else w_div/sum(w_div)

# --- Blends del subset ---
# tmean por fila (quita min y max si hay >=3)
subset_tmean <- apply(P_sub, 1, function(v){
  v <- v[is.finite(v)]
  if (length(v) >= 3) { v <- sort(v); mean(v[-c(1,length(v))]) }
  else if (length(v) > 0) mean(v) else NA_real_
})

# logit-mean ponderado por diversidad (renormaliza por fila para NAs)
logit  <- function(p) log(p/(1-p)); ilogit <- function(z) 1/(1+exp(-z))
Z  <- logit(Pclip)
W  <- matrix(w_div, nrow=nrow(Z), ncol=ncol(Z), byrow=TRUE)
W[!is.finite(Z)] <- 0
wsum <- rowSums(W)
W[wsum>0,] <- W[wsum>0,] / wsum[wsum>0]
subset_logit_div <- ilogit(rowSums(Z * W))

# Guardar archivos FULL del subset
fwrite(data.table(numero_de_cliente = ALL$numero_de_cliente, prob = subset_tmean),
       "ens_subset_tmean_FULL_prediccion.txt", sep="\t")
fwrite(data.table(numero_de_cliente = ALL$numero_de_cliente, prob = subset_logit_div),
       "ens_subset_logit_div_FULL_prediccion.txt", sep="\t")

cat("✓ Generados:\n",
    "  - ens_subset_tmean_FULL_prediccion.txt\n",
    "  - ens_subset_logit_div_FULL_prediccion.txt\n", sep="")


Encontrados 16 archivos.
Seleccionados 8 modelos de 16 
    model score
   <char> <num>
1:     p6 0.470
2:    p14 0.470
3:     p8 0.464
4:    p16 0.464
5:     p5 0.463
6:    p13 0.463
7:     p7 0.459
8:    p15 0.459
Referencia: ens_logit_WF_30_19_18_17_14_prediccion.txt  | n= 165093 
✓ Generados:
  - ens_subset_tmean_FULL_prediccion.txt
  - ens_subset_logit_div_FULL_prediccion.txt


In [21]:
# ========== SUBMIT (subset) K=10500/11000/11500 ==========
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

FILES <- c("ens_subset_tmean_FULL_prediccion.txt",
           "ens_subset_logit_div_FULL_prediccion.txt")
NAMES <- c("subset_tmean", "subset_logit_div")
K_SET <- c(10500L, 11000L, 11500L)

COMPETITION   <- "data-mining-analista-sr-2025-a"
EXP_NAME_BASE <- "KA_SUBSET_FULL"

if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent=TRUE)
  if (!inherits(P,"try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION   <- P$kaggle$competencia
    if (!is.null(P$experimento))        EXP_NAME_BASE <- paste0(P$experimento, "_SUBSET_FULL")
  }
}

stopifnot(!inherits(try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE),"try-error"))
dir.create("kaggle", showWarnings = FALSE)

for (i in seq_along(FILES)) {
  f <- FILES[i]; nm <- NAMES[i]
  if (!file.exists(f)) { warning("Falta ", f); next }
  ens <- fread(f)
  if (!all(c("numero_de_cliente","prob") %in% names(ens)))
    setnames(ens, names(ens)[1:2], c("numero_de_cliente","prob"))
  setorder(ens, -prob, numero_de_cliente)

  for (k in K_SET) {
    sub <- ens[, .(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= k))]
    out <- sprintf("kaggle/%s_%s_top_%d.csv", EXP_NAME_BASE, nm, k)
    fwrite(sub, out)
    msg <- sprintf("SUBSET FULL | %s | top=%d", nm, k)
    cat("Subiendo:", out, "...\n")
    res <- system2("kaggle",
                   c("competitions","submit","-c", COMPETITION, "-f", out, "-m", shQuote(msg)),
                   stdout=TRUE, stderr=TRUE)
    cat(paste(res, collapse="\n"), "\n")
    Sys.sleep(8)  # mini-pausa anti 429
  }
}
cat("Listo. Revisá el LB y nos quedamos con el mejor.\n")


Subiendo: kaggle/KA_SUBSET_FULL_subset_tmean_top_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.36MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_SUBSET_FULL_subset_tmean_top_11000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.38MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_SUBSET_FULL_subset_tmean_top_11500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.21MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_SUBSET_FULL_subset_logit_div_top_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 2.86MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_SUBSET_FULL_subset_logit_div_top_11000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.36MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_SUBSET_FULL_subset_logit_div_top_11500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.18MB/s]
Su

In [22]:
# ====== BLENDS FULL "CREATIVOS" (RRF / SoftmaxRank / PowerMean3 / Q75) ======
if (!require(data.table)) install.packages("data.table"); library(data.table)

# Bases donde buscar WFxxxx/prediccion.txt (ajustá EXP_BASE si hace falta)
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),           # ej: "/home/juaniripoll27/buckets/b1/exp"
  Sys.glob("/home/juaniripoll27/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES)>0)

# Recolecto todos los prediccion.txt dentro de carpetas WFxxxx
paths <- unique(unlist(lapply(BASES, function(b)
  list.files(b, pattern="^prediccion\\.(txt|csv)$", recursive=TRUE, full.names=TRUE))))
dirs  <- basename(dirname(paths))
paths <- paths[grepl("^WF\\d+$|^WF95\\d{2}$", dirs)]
stopifnot(length(paths)>0)
cat("Detectados", length(paths), "archivos.\n")

# Lector normalizado
read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, paths, seq_along(paths))

# Base de referencia FULL (garantiza universo correcto para Kaggle)
ref_candidates <- c(
  "ens_logit_WF_30_19_18_17_14_prediccion.txt",  # alguna que ya haya pasado
  paths[1]
)
ref_path <- ref_candidates[file.exists(ref_candidates)][1]
stopifnot(length(ref_path)>0)
REF <- fread(ref_path)[, .(numero_de_cliente)]
cat("Referencia:", ref_path, " | n=", nrow(REF), "\n")

# Unión sobre referencia
union_all <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=TRUE), DTs)
ALL <- merge(REF, union_all, by="numero_de_cliente", all.x=TRUE)
pred_cols <- grep("^p\\d+$", names(ALL), value=TRUE)
P <- as.matrix(ALL[, ..pred_cols])                  # N x M (con NAs)
N <- nrow(P); M <- ncol(P)

# Utils
clip01 <- function(x, eps=1e-6) pmin(pmax(x, eps), 1-eps)
logit  <- function(p) log(p/(1-p))
ilogit <- function(z) 1/(1+exp(-z))

# ---------- 1) RRF (Reciprocal Rank Fusion) ----------
# rank 1 = mejor (desc), saltando NAs por columna
rank_desc <- function(v){
  r <- rep(NA_real_, length(v)); ok <- is.finite(v)
  r[ok] <- rank(-v[ok], ties.method="average")     # menor = mejor
  r
}
R <- do.call(cbind, lapply(seq_len(M), function(j) rank_desc(P[,j])))
C <- 60  # constante típica en IR (ajustable)
S_rrf <- rowSums(1/(C + R), na.rm=TRUE)

# ---------- 2) Softmax-Rank ----------
# normalizo ranks a [0,1] y aplico exp(alpha*(1 - r_norm))
alpha <- 10
Rnorm <- sweep(R, 2, apply(R, 2, function(x) max(x, na.rm=TRUE)), "/")
Rnorm[!is.finite(Rnorm)] <- NA_real_
S_smax <- rowMeans(exp(alpha*(1 - Rnorm)), na.rm=TRUE)

# ---------- 3) Power-Mean (gamma = 3) ----------
gamma <- 3
Pclip <- clip01(P)
S_pmean3 <- (rowMeans(Pclip^gamma, na.rm=TRUE))^(1/gamma)

# ---------- 4) Q75 (cuantil 75% por fila) ----------
S_q75 <- apply(P, 1, function(v){
  v <- v[is.finite(v)]
  if (length(v)) quantile(v, 0.75, names=FALSE, type=7) else NA_real_
})

# Guardar como prediccion.txt
save_pred <- function(name, s){
  fname <- paste0("ens_creative_", name, "_FULL_prediccion.txt")
  fwrite(data.table(numero_de_cliente = ALL$numero_de_cliente, prob = s), fname, sep="\t")
  cat("✓", fname, "\n")
}
save_pred("rrf",    S_rrf)
save_pred("smax",   S_smax)
save_pred("pmean3", S_pmean3)
save_pred("q75",    S_q75)


Detectados 16 archivos.
Referencia: ens_logit_WF_30_19_18_17_14_prediccion.txt  | n= 165093 
✓ ens_creative_rrf_FULL_prediccion.txt 
✓ ens_creative_smax_FULL_prediccion.txt 
✓ ens_creative_pmean3_FULL_prediccion.txt 
✓ ens_creative_q75_FULL_prediccion.txt 


In [23]:
# ====== SUBMIT de blends creativos (K=10500/11000/11500) ======
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

BLENDS <- c("rrf","smax","pmean3","q75")   # podés comentar los que no quieras
K_SET  <- c(10500L, 11000L, 11500L)

COMPETITION   <- "data-mining-analista-sr-2025-a"
EXP_NAME_BASE <- "KA_CREATIVE_FULL"
if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent=TRUE)
  if (!inherits(P,"try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION   <- P$kaggle$competencia
    if (!is.null(P$experimento))        EXP_NAME_BASE <- paste0(P$experimento, "_CREATIVE_FULL")
  }
}
stopifnot(!inherits(try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE),"try-error"))
dir.create("kaggle", showWarnings = FALSE)

for (b in BLENDS) {
  pred <- sprintf("ens_creative_%s_FULL_prediccion.txt", b)
  if (!file.exists(pred)) { warning("Falta ", pred); next }
  ens <- fread(pred)
  if (!all(c("numero_de_cliente","prob") %in% names(ens)))
    setnames(ens, names(ens)[1:2], c("numero_de_cliente","prob"))
  setorder(ens, -prob, numero_de_cliente)

  for (k in K_SET) {
    sub <- ens[, .(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= k))]
    out <- sprintf("kaggle/%s_%s_top_%d.csv", EXP_NAME_BASE, b, k)
    fwrite(sub, out)
    msg <- sprintf("CREATIVE FULL | blend=%s | top=%d", b, k)
    cat("Subiendo:", out, "...\n")
    res <- system2("kaggle",
                   c("competitions","submit","-c", COMPETITION, "-f", out, "-m", shQuote(msg)),
                   stdout=TRUE, stderr=TRUE)
    cat(paste(res, collapse="\n"), "\n")
    Sys.sleep(8)  # mini pausa contra 429
  }
}
cat("Listo. Mirá si alguno supera tmean@11000 (66.017).\n")


Subiendo: kaggle/KA_CREATIVE_FULL_rrf_top_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.57MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_CREATIVE_FULL_rrf_top_11000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.24MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_CREATIVE_FULL_rrf_top_11500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.61MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_CREATIVE_FULL_smax_top_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.48MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_CREATIVE_FULL_smax_top_11000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.60MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_CREATIVE_FULL_smax_top_11500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.65MB/s]
Successfully submitted to Data Mining, Analista Sr 20

In [26]:
# ================= CONSENSO (votes) + FILL (tmean) con 8 WFs =================
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

WF_IDS    <- c(30,19,18,17,14,13,12)
WF_LABELS <- sprintf("WF95%02d", WF_IDS)

# --- dónde buscar los prediccion.txt ---
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),        # ej: /home/juaniripoll27/buckets/b1/exp
  Sys.glob("/home/juaniripoll27/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES)>0)

find_pred <- function(wf) {
  for (b in BASES) {
    p <- file.path(b, wf, "prediccion.txt")
    if (file.exists(p)) return(normalizePath(p))
  }
  NA_character_
}
FILES <- vapply(WF_LABELS, find_pred, FUN.VALUE = character(1))
if (anyNA(FILES)) stop("Faltan:\n  - ",
  paste(file.path(WF_LABELS[is.na(FILES)], "prediccion.txt"), collapse="\n  - "))

cat("Ensamblo con:\n", paste(" •", FILES, collapse="\n"), "\n")

# --- leo y normalizo cada pred ---
read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, FILES, seq_along(FILES))

# --- referencia FULL (garantiza universo que Kaggle espera) ---
ref_candidates <- c(
  "ens_logit_WF_30_19_18_17_14_prediccion.txt",  # alguna que ya pasó
  FILES[1]
)
ref_path <- ref_candidates[file.exists(ref_candidates)][1]
stopifnot(length(ref_path)>0)
REF <- fread(ref_path)[, .(numero_de_cliente)]
cat("Referencia:", ref_path, " | n=", nrow(REF), "\n")

# --- union sobre referencia ---
UNION <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=TRUE), DTs)
ALL   <- merge(REF, UNION, by="numero_de_cliente", all.x=TRUE)
pred_cols <- grep("^p\\d+$", names(ALL), value=TRUE)

# --- métricas por fila ---
# 1) votos: cuántos modelos te dejan en su top-L (desc)
L_TOP <- 12000L   # podés probar 11500 o 12500 si querés
rank_desc <- function(v){
  r <- rep(NA_real_, length(v)); ok <- is.finite(v)
  r[ok] <- rank(-v[ok], ties.method="average")
  r
}
R <- do.call(cbind, lapply(pred_cols, function(cn) rank_desc(ALL[[cn]])))
votes <- rowSums(R <= L_TOP, na.rm=TRUE)

# 2) tmean: quita min y max (robusto)
tmean_row <- function(v){
  vv <- v[is.finite(v)]
  if (length(vv) >= 3) { vv <- sort(vv); mean(vv[-c(1,length(vv))]) }
  else if (length(vv) > 0) mean(vv) else NA_real_
}
tmean <- apply(as.matrix(ALL[, ..pred_cols]), 1, tmean_row)

# 3) rank promedio (para último desempate)
rmean <- rowMeans(R, na.rm=TRUE)

# --- score y orden final: votes ↓ , tmean ↓ , rmean ↑ ---
S <- data.table(
  numero_de_cliente = ALL$numero_de_cliente,
  votes = votes,
  tmean = tmean,
  rmean = rmean
)
setorder(S, -votes, -tmean, rmean)

# --- arma y (opcional) sube submits para K = 10500 / 11000 / 11500 ---
# lee competition/cortes desde PARAM.yml si existe
COMPETITION <- "data-mining-analista-sr-2025-a"
if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent=TRUE)
  if (!inherits(P,"try-error") && !is.null(P$kaggle$competencia)) COMPETITION <- P$kaggle$competencia
}

dir.create("kaggle", showWarnings = FALSE)
TAG <- paste(WF_IDS, collapse="_")
BLEND_NAME <- sprintf("consensus%02d_tmean", length(pred_cols))

K_SET <- c(10500L, 11000L, 11500L)
OUTS <- character()
for (K in K_SET) {
  sub <- S[, .(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= K))]
  fname <- sprintf("kaggle/KA_%s_wf%s_L%d_top_%d.csv", BLEND_NAME, TAG, L_TOP, K)
  fwrite(sub, fname)
  OUTS <- c(OUTS, fname)
}
cat("✅ Generados:\n  ", paste(OUTS, collapse="\n  "), "\n", sep="")

# --- toggle para subir automáticamente a Kaggle ---
SUBMIT <- TRUE  # ponelo TRUE si querés que suba ahora
if (SUBMIT) {
  # pausa corta anti-429
  stopifnot(!inherits(try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE),"try-error"))
  for (f in OUTS) {
    msg <- sprintf("WFs %s | consensus(core=L%d by votes) + tmean fill | %s", TAG, L_TOP, basename(f))
    cat("Subiendo:", f, "...\n")
    res <- system2("kaggle", c("competitions","submit","-c", COMPETITION, "-f", f, "-m", shQuote(msg)),
                   stdout=TRUE, stderr=TRUE)
    cat(paste(res, collapse="\n"), "\n"); Sys.sleep(8)
  }
}


Ensamblo con:
  • /home/juaniripoll27/buckets/b1/exp/WF9530/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9519/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9518/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9517/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9514/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9513/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9512/prediccion.txt 
Referencia: ens_logit_WF_30_19_18_17_14_prediccion.txt  | n= 165093 
✅ Generados:
  kaggle/KA_consensus07_tmean_wf30_19_18_17_14_13_12_L12000_top_10500.csv
  kaggle/KA_consensus07_tmean_wf30_19_18_17_14_13_12_L12000_top_11000.csv
  kaggle/KA_consensus07_tmean_wf30_19_18_17_14_13_12_L12000_top_11500.csv
Subiendo: kaggle/KA_consensus07_tmean_wf30_19_18_17_14_13_12_L12000_top_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.57MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_consensus07_tmean_wf30_19_18_17_14_13_12_L1

In [28]:
# ==================== BEST-8 ENSEMBLES (FULL) + SUBMITS ====================
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

# --- 1) Cargar los 8 mejores WFs -------------------------------------------------
WF_IDS    <- c(30,19,18,17,14,13,12)
WF_LABELS <- sprintf("WF95%02d", WF_IDS)

BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),          # ej: /home/juaniripoll27/buckets/b1/exp
  Sys.glob("/home/juaniripoll27/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES)>0)

find_pred <- function(wf){
  for (b in BASES) {
    p <- file.path(b, wf, "prediccion.txt")
    if (file.exists(p)) return(normalizePath(p))
  }
  NA_character_
}
FILES <- vapply(WF_LABELS, find_pred, FUN.VALUE=character(1))
if (anyNA(FILES)) stop("Faltan:\n  - ", paste(file.path(WF_LABELS[is.na(FILES)], "prediccion.txt"), collapse="\n  - "))
cat("Uso estos 8:\n", paste(" •", FILES, collapse="\n"), "\n")

read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, FILES, seq_along(FILES))

# --- 2) Referencia FULL (garantiza universo Kaggle) ------------------------------
ref_candidates <- c(
  "ens_logit_WF_30_19_18_17_14_prediccion.txt",  # alguna que ya pasó
  FILES[1]
)
ref_path <- ref_candidates[file.exists(ref_candidates)][1]
stopifnot(length(ref_path)>0)
REF <- fread(ref_path)[, .(numero_de_cliente)]
cat("Referencia:", ref_path, " | n=", nrow(REF), "\n")

UNION <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=TRUE), DTs)
ALL   <- merge(REF, UNION, by="numero_de_cliente", all.x=TRUE)
pred_cols <- grep("^p\\d+$", names(ALL), value=TRUE)
P <- as.matrix(ALL[, ..pred_cols])     # N x 8 (con NAs)
N <- nrow(P); M <- ncol(P)

# --- utils ----------------------------------------------------------------------
scale01 <- function(x){ rng <- range(x, na.rm=TRUE); if (diff(rng)==0) rep(0.5,length(x)) else (x-rng[1])/diff(rng) }
clip01  <- function(x, eps=1e-6) pmin(pmax(x, eps), 1-eps)
logit   <- function(p) log(p/(1-p))
ilogit  <- function(z) 1/(1+exp(-z))
rank_desc <- function(v){ r <- rep(NA_real_, length(v)); ok <- is.finite(v); r[ok] <- rank(-v[ok], ties.method="average"); r }

# --- 3) Blends de ranking y prob -------------------------------------------------
# ranks (1 = mejor)
R <- do.call(cbind, lapply(seq_len(M), function(j) rank_desc(P[,j])))

# 3.1 promedios de prob
tmean_row <- function(v){
  vv <- v[is.finite(v)]
  if (length(vv) >= 3) { vv <- sort(vv); mean(vv[-c(1,length(vv))]) }
  else if (length(vv) > 0) mean(vv) else NA_real_
}
S_tmean  <- apply(P, 1, tmean_row)
S_mean   <- rowMeans(P, na.rm=TRUE)
S_median <- apply(P, 1, function(v){ v <- v[is.finite(v)]; if(length(v)) median(v) else NA_real_ })

# 3.2 rank-mean (Borda simple)
S_rankmean <- -rowMeans(R, na.rm=TRUE)

# 3.3 logit y logit_div (pesos por diversidad)
Pclip <- clip01(P); Z <- logit(Pclip)
# diversidad desde la intersección
inter <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=FALSE), DTs)
pred_cols_inter <- grep("^p\\d+$", names(inter), value=TRUE)
Minter <- as.matrix(inter[, ..pred_cols_inter])
C <- suppressWarnings(cor(Minter, use="pairwise.complete.obs"))
avg_corr <- (rowSums(C) - 1) / pmax(1, (ncol(C)-1))
w_div <- pmax(0, 1 - avg_corr); w_div <- w_div / sum(w_div)
# map a FULL
map_w <- rep(min(w_div)/10, M); names(map_w) <- pred_cols; names(w_div) <- pred_cols_inter; map_w[names(w_div)] <- w_div
W <- matrix(map_w, nrow=N, ncol=M, byrow=TRUE); W[!is.finite(Z)] <- 0; wsum <- rowSums(W); W[wsum>0,] <- W[wsum>0,]/wsum[wsum>0]
S


Uso estos 8:
  • /home/juaniripoll27/buckets/b1/exp/WF9530/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9519/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9518/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9517/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9514/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9513/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9512/prediccion.txt 
Referencia: ens_logit_WF_30_19_18_17_14_prediccion.txt  | n= 165093 


numero_de_cliente,votes,tmean,rmean
<int>,<dbl>,<dbl>,<dbl>
79115621,7,0.8087969,139.42857
103178792,7,0.7295293,107.00000
140404349,7,0.7127667,17.42857
127755448,7,0.6967450,24.28571
87971010,7,0.6887815,49.57143
71804832,7,0.6881719,49.00000
66858028,7,0.6836464,503.42857
74339852,7,0.6763476,56.57143
141368357,7,0.6653142,96.00000


In [29]:
# ================== 7-WF ENSEMBLES (FULL) + SUBMITS (10500/11000/11500) ==================
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

WF_IDS    <- c(30,19,18,17,14,13,12)              # 9510 NO está; 7 modelos
WF_LABELS <- sprintf("WF95%02d", WF_IDS)

# Bases donde buscar prediccion.txt
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),   # ej: "/home/juaniripoll27/buckets/b1/exp"
  Sys.glob("/home/juaniripoll27/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES)>0)

find_pred <- function(wf){
  for (b in BASES) {
    p <- file.path(b, wf, "prediccion.txt")
    if (file.exists(p)) return(normalizePath(p))
  }
  NA_character_
}
FILES <- vapply(WF_LABELS, find_pred, FUN.VALUE=character(1))
if (anyNA(FILES)) stop("Faltan:\n  - ",
  paste(file.path(WF_LABELS[is.na(FILES)], "prediccion.txt"), collapse="\n  - "))

cat("Uso estos 7 WFs:\n", paste(" •", FILES, collapse="\n"), "\n")

read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, FILES, seq_along(FILES))

# Referencia FULL (misma población que Kaggle)
ref_candidates <- c(
  "ens_logit_WF_30_19_18_17_14_prediccion.txt",   # ya pasó
  FILES[1]
)
ref_path <- ref_candidates[file.exists(ref_candidates)][1]
stopifnot(length(ref_path)>0)
REF <- fread(ref_path)[, .(numero_de_cliente)]
cat("Referencia:", ref_path, " | n=", nrow(REF), "\n")

UNION <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=TRUE), DTs)
ALL   <- merge(REF, UNION, by="numero_de_cliente", all.x=TRUE)
pred_cols <- grep("^p\\d+$", names(ALL), value=TRUE)

# Matrix de probs
P <- as.matrix(ALL[, ..pred_cols]); N <- nrow(P); M <- ncol(P)

# Utils
clip01  <- function(x, eps=1e-6) pmin(pmax(x, eps), 1-eps)
scale01 <- function(x){ r <- range(x, na.rm=TRUE); if (diff(r)==0) rep(0.5,length(x)) else (x-r[1])/diff(r) }
rank_desc <- function(v){ r <- rep(NA_real_, length(v)); ok <- is.finite(v); r[ok] <- rank(-v[ok], ties.method="average"); r }
logit   <- function(p) log(p/(1-p)); ilogit <- function(z) 1/(1+exp(-z))

# Ranks (1=mejor)
R <- do.call(cbind, lapply(seq_len(M), function(j) rank_desc(P[,j])))

# --- Fusiones ---
# 1) tmean (recorta min y max por fila)
tmean_row <- function(v){
  vv <- v[is.finite(v)]
  if (length(vv) >= 3) { vv <- sort(vv); mean(vv[-c(1,length(vv))]) }
  else if (length(vv) > 0) mean(vv) else NA_real_
}
S_tmean <- apply(P, 1, tmean_row)

# 2) winsor (clip fila a [p10,p90] y promedio)
S_winsor <- apply(P, 1, function(v){
  vv <- v[is.finite(v)]
  if (!length(vv)) return(NA_real_)
  qs <- quantile(vv, c(0.10,0.90), names=FALSE, type=7)
  vv <- pmin(pmax(vv, qs[1]), qs[2])
  mean(vv)
})

# 3) geomrank (geom. mean de score de ranking s = 1 - rank_norm)
Rmax <- apply(R, 2, function(x) max(x, na.rm=TRUE))
S_geomrank <- apply(R, 1, function(r){
  ok <- is.finite(r)
  if (!any(ok)) return(NA_real_)
  s <- 1 - (r[ok]-1)/(Rmax[ok]-1)          # [0,1], 1=mejor
  exp(mean(log(pmax(s, 1e-9))))
})

# 4) híbrido 0.6*rankmean + 0.4*tmean (ambos en [0,1])
S_rankmean <- -rowMeans(R, na.rm=TRUE)
hyb60 <- 0.6*scale01(S_rankmean) + 0.4*scale01(S_tmean)

# 5-6) votos en top-L con desempate (tmean)
votes_score <- function(L){
  v <- rowSums(R <= L, na.rm=TRUE)
  v + 1e-3*scale01(S_tmean) + 1e-6*(-rowMeans(R, na.rm=TRUE))
}
S_votesL11000 <- votes_score(11000L)
S_votesL10000 <- votes_score(10000L)

# (opcional) logit/logit_div por si querés comparar después:
# Pclip <- clip01(P); Z <- logit(Pclip)
# S_logit <- ilogit(rowMeans(Z, na.rm=TRUE))

# Normalizo todos a [0,1] para usarlos como 'prob' y guardo
BLENDS <- list(
  tmean       = scale01(S_tmean),
  winsor      = scale01(S_winsor),
  geomrank    = scale01(S_geomrank),
  hyb60       = scale01(hyb60),
  votesL11000 = scale01(S_votesL11000),
  votesL10000 = scale01(S_votesL10000)
)

for (nm in names(BLENDS)) {
  fn <- sprintf("ens_top7_%s_FULL_prediccion.txt", nm)
  fwrite(data.table(numero_de_cliente = ALL$numero_de_cliente, prob = BLENDS[[nm]]), fn, sep="\t")
  cat("✓", fn, "\n")
}

# ---------------- SUBMITS (toggle) ----------------
SUBMIT <- TRUE                          # poné FALSE si querés evitar subir ahora
BLENDS_TO_SUBMIT <- c("tmean","winsor","geomrank","hyb60","votesL11000","votesL10000")
K_SET <- c(10500L, 11000L, 11500L)

COMPETITION   <- "data-mining-analista-sr-2025-a"
EXP_NAME_BASE <- "KA_TOP7"
if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent=TRUE)
  if (!inherits(P,"try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION   <- P$kaggle$competencia
    if (!is.null(P$experimento))        EXP_NAME_BASE <- paste0(P$experimento, "_TOP7")
  }
}

if (SUBMIT) {
  stopifnot(!inherits(try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE),"try-error"))
  dir.create("kaggle", showWarnings = FALSE)
  for (nm in BLENDS_TO_SUBMIT) {
    pred <- sprintf("ens_top7_%s_FULL_prediccion.txt", nm)
    if (!file.exists(pred)) next
    ens <- fread(pred); setnames(ens, names(ens)[1:2], c("numero_de_cliente","prob"))
    setorder(ens, -prob, numero_de_cliente)
    for (k in K_SET) {
      sub <- ens[, .(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= k))]
      out <- sprintf("kaggle/%s_%s_top_%d.csv", EXP_NAME_BASE, nm, k)
      fwrite(sub, out)
      msg <- sprintf("TOP7 | blend=%s | top=%d", nm, k)
      cat("Subiendo:", out, "...\n")
      res <- system2("kaggle",
        c("competitions","submit","-c", COMPETITION, "-f", out, "-m", shQuote(msg)),
        stdout=TRUE, stderr=TRUE)
      cat(paste(res, collapse="\n"), "\n")
      Sys.sleep(8)  # mini pausa anti 429
    }
  }
  cat("Listo. Revisá si alguno supera 66.017.\n")
}


Uso estos 7 WFs:
  • /home/juaniripoll27/buckets/b1/exp/WF9530/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9519/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9518/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9517/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9514/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9513/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9512/prediccion.txt 
Referencia: ens_logit_WF_30_19_18_17_14_prediccion.txt  | n= 165093 
✓ ens_top7_tmean_FULL_prediccion.txt 
✓ ens_top7_winsor_FULL_prediccion.txt 
✓ ens_top7_geomrank_FULL_prediccion.txt 
✓ ens_top7_hyb60_FULL_prediccion.txt 
✓ ens_top7_votesL11000_FULL_prediccion.txt 
✓ ens_top7_votesL10000_FULL_prediccion.txt 
Subiendo: kaggle/KA_TOP7_tmean_top_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.59MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_TOP7_tmean_top_11000.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:

In [31]:
# ====== DISRUPTIVO: CONSENSO + TOPNESS + MULTI-L + META-RANK (7 WFs) ======
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

# --- WFs a usar (9510 no tiene prediccion.txt) ---
WF_IDS    <- c(30,19,18,17,14,13,12)
WF_LABELS <- sprintf("WF95%02d", WF_IDS)

# --- Dónde buscar ---
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),
  Sys.glob("/home/*/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES)>0)

find_pred <- function(wf){
  for (b in BASES) {
    p <- file.path(b, wf, "prediccion.txt")
    if (file.exists(p)) return(normalizePath(p))
  }
  NA_character_
}
FILES <- vapply(WF_LABELS, find_pred, FUN.VALUE=character(1))
if (anyNA(FILES)) stop("Faltan predicciones:\n  - ",
                       paste(file.path(WF_LABELS[is.na(FILES)], "prediccion.txt"), collapse="\n  - "))
cat("Ensamblo con:\n", paste(" •", FILES, collapse="\n"), "\n")

# --- Lectura normalizada ---
read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, FILES, seq_along(FILES))

# --- Referencia FULL (misma población que Kaggle) ---
ref_candidates <- c(
  "ens_tmean_FULL_prediccion.txt",                  # generado antes (ALL FULL)
  "ens_logit_WF_30_19_18_17_14_prediccion.txt",
  FILES[1]
)
ref_path <- ref_candidates[file.exists(ref_candidates)][1]
stopifnot(length(ref_path)>0)
REF <- fread(ref_path)[, .(numero_de_cliente)]
cat("Referencia:", ref_path, " | n=", nrow(REF), "\n")

# --- Unión FULL ---
UNION <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=TRUE), DTs)
ALL   <- merge(REF, UNION, by="numero_de_cliente", all.x=TRUE)
pred_cols <- grep("^p\\d+$", names(ALL), value=TRUE)
P <- as.matrix(ALL[, ..pred_cols]); N <- nrow(P); M <- ncol(P)

# --- Utils ---
scale01 <- function(x){ r <- range(x, na.rm=TRUE); if (diff(r)==0) rep(0.5,length(x)) else (x-r[1])/diff(r) }
rank_desc <- function(v){ r <- rep(NA_real_, length(v)); ok <- is.finite(v); r[ok] <- rank(-v[ok], ties.method="average"); r }

# --- Matriz de ranks (1=mejor) ---
R <- do.call(cbind, lapply(seq_len(M), function(j) rank_desc(P[,j])))

# =====================================================================
# 1) votesL11000 con desempate por TOPNESS (FIX: sin pmax que aplana)
# =====================================================================
L_BASE <- 11000L
votes  <- rowSums(R <= L_BASE, na.rm=TRUE)

Z <- (L_BASE - R) / L_BASE     # misma forma que R
Z[!is.finite(Z)] <- NA_real_
Z[Z < 0] <- 0                  # recorte a 0 sin perder dimensión
topness <- rowSums(Z, na.rm=TRUE)

S_votesL11000_topness <- votes + 0.01*scale01(topness) + 1e-6*(-rowMeans(R, na.rm=TRUE))

# =============================================================
# 2) votesMulti y 3) topnessMulti en L = {10000,10500,11000,11500}
# =============================================================
LSET <- c(10000L,10500L,11000L,11500L)
W    <- c(6,4,3,2)   # más peso a L chicos

# votos por L (N x |LSET|)
Vm <- sapply(LSET, function(L) rowSums(R <= L, na.rm=TRUE))
S_votesMulti <- rowSums(sweep(Vm, 2, W, `*`), na.rm=TRUE) + 1e-3*scale01(rowMeans(P, na.rm=TRUE))

# topness por L (N x |LSET|), sin pmax
Tm <- sapply(LSET, function(L) {
  ZL <- (L - R) / L
  ZL[!is.finite(ZL)] <- NA_real_
  ZL[ZL < 0] <- 0
  rowSums(ZL, na.rm=TRUE)
})
S_topnessMulti <- rowSums(sweep(Tm, 2, W, `*`), na.rm=TRUE)

# ============================================================
# 4) META-RANK: votesL11000_topness + tmean_FULL + q75_FULL
# ============================================================
# Asegurate de haber generado estos antes:
tmean_full_path <- "ens_tmean_FULL_prediccion.txt"
q75_full_path   <- "ens_creative_q75_FULL_prediccion.txt"

if (!file.exists(tmean_full_path) || !file.exists(q75_full_path))
  stop("Faltan 'ens_tmean_FULL_prediccion.txt' y/o 'ens_creative_q75_FULL_prediccion.txt'.")

S_tmeanAll <- fread(tmean_full_path)[, prob]
S_q75All   <- fread(q75_full_path)[, prob]
stopifnot(length(S_tmeanAll)==N, length(S_q75All)==N)

rank_desc_vec <- function(v){ rank(-v, ties.method="average") }
RB <- cbind(rank_desc_vec(S_votesL11000_topness),
            rank_desc_vec(S_tmeanAll),
            rank_desc_vec(S_q75All))

S_meta_borda <- -rowMeans(RB, na.rm=TRUE)

alpha <- 10
RBn <- sweep(RB, 2, apply(RB, 2, max, na.rm=TRUE), "/")
RBn[!is.finite(RBn)] <- NA_real_
S_meta_smax <- rowMeans(exp(alpha*(1 - RBn)), na.rm=TRUE)

# --- Guardar todos como prob en [0,1] ---
BLENDS <- list(
  votesL11000_topness = scale01(S_votesL11000_topness),
  votesMulti          = scale01(S_votesMulti),
  topnessMulti        = scale01(S_topnessMulti),
  meta_borda          = scale01(S_meta_borda),
  meta_smax           = scale01(S_meta_smax)
)
for (nm in names(BLENDS)) {
  fn <- sprintf("ens_disrupt_%s_FULL_prediccion.txt", nm)
  fwrite(data.table(numero_de_cliente = ALL$numero_de_cliente, prob = BLENDS[[nm]]), fn, sep="\t")
  cat("✓", fn, "\n")
}

# ===================== SUBMITS (toggle) =====================
SUBMIT <- TRUE
BLENDS_TO_SUBMIT <- c("votesL11000_topness","votesMulti","topnessMulti","meta_borda","meta_smax")
K_SET <- c(10500L,11000L,11500L)

COMPETITION   <- "data-mining-analista-sr-2025-a"
EXP_NAME_BASE <- "KA_DISRUPT_TOP7"
if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent=TRUE)
  if (!inherits(P,"try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION   <- P$kaggle$competencia
    if (!is.null(P$experimento))        EXP_NAME_BASE <- paste0(P$experimento, "_DISRUPT_TOP7")
  }
}

if (SUBMIT) {
  stopifnot(!inherits(try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE),"try-error"))
  dir.create("kaggle", showWarnings = FALSE)
  for (nm in BLENDS_TO_SUBMIT) {
    pred <- sprintf("ens_disrupt_%s_FULL_prediccion.txt", nm)
    if (!file.exists(pred)) next
    ens <- fread(pred); setnames(ens, names(ens)[1:2], c("numero_de_cliente","prob"))
    setorder(ens, -prob, numero_de_cliente)
    for (k in K_SET) {
      out <- sprintf("kaggle/%s_%s_top_%d.csv", EXP_NAME_BASE, nm, k)
      sub <- ens[,.(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= k))]
      fwrite(sub, out)
      msg <- sprintf("DISRUPT | %s | top=%d", nm, k)
      cat("Subiendo:", out, "...\n")
      res <- system2("kaggle",
                     c("competitions","submit","-c", COMPETITION, "-f", out, "-m", shQuote(msg)),
                     stdout=TRUE, stderr=TRUE)
      cat(paste(res, collapse="\n"), "\n")
      Sys.sleep(8)  # pausa anti 429
    }
  }
  cat("Listo. Revisá si alguno supera 66.257.\n")
}


Ensamblo con:
  • /home/juaniripoll27/buckets/b1/exp/WF9530/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9519/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9518/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9517/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9514/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9513/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9512/prediccion.txt 
Referencia: ens_tmean_FULL_prediccion.txt  | n= 165093 
✓ ens_disrupt_votesL11000_topness_FULL_prediccion.txt 
✓ ens_disrupt_votesMulti_FULL_prediccion.txt 
✓ ens_disrupt_topnessMulti_FULL_prediccion.txt 
✓ ens_disrupt_meta_borda_FULL_prediccion.txt 
✓ ens_disrupt_meta_smax_FULL_prediccion.txt 
Subiendo: kaggle/KA_DISRUPT_TOP7_votesL11000_topness_top_10500.csv ...
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.47MB/s]
Successfully submitted to Data Mining, Analista Sr 2025A 
Subiendo: kaggle/KA_DISRUPT_TOP7_votesL11000_topness_top_11000.csv ...
100%|██████████

In [34]:
# ================== ANCHOR (VOTES) + FILL (tmean / FULL / q75 / OR) ==================
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

WF_IDS    <- c(30,19,18,17,14,13,12)                 # 7 modelos buenos (sin 9510)
WF_LABELS <- sprintf("WF95%02d", WF_IDS)

# --- localizar prediccion.txt ---
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),
  Sys.glob("/home/*/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES)>0)

find_pred <- function(wf){
  for (b in BASES) {
    p <- file.path(b, wf, "prediccion.txt")
    if (file.exists(p)) return(normalizePath(p))
  }
  NA_character_
}
FILES <- vapply(WF_LABELS, find_pred, FUN.VALUE=character(1))
if (anyNA(FILES)) stop("Faltan:\n  - ", paste(file.path(WF_LABELS[is.na(FILES)], "prediccion.txt"), collapse="\n  - "))
cat("WF usados:\n", paste(" •", FILES, collapse="\n"), "\n")

# --- lecturas ---
read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, FILES, seq_along(FILES))

# --- referencia FULL (misma población) ---
ref_candidates <- c(
  "ens_tmean_FULL_prediccion.txt",
  "ens_logit_WF_30_19_18_17_14_prediccion.txt",
  FILES[1]
)
ref_path <- ref_candidates[file.exists(ref_candidates)][1]
stopifnot(length(ref_path)>0)
REF <- fread(ref_path)[, .(numero_de_cliente)]
cat("Referencia:", ref_path, " | n=", nrow(REF), "\n")

# --- unión FULL de los 7 ---
UNION <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=TRUE), DTs)
ALL   <- merge(REF, UNION, by="numero_de_cliente", all.x=TRUE)
pred_cols <- grep("^p\\d+$", names(ALL), value=TRUE)
P <- as.matrix(ALL[, ..pred_cols]); N <- nrow(P); M <- ncol(P)

# ---- utils ----
clip01  <- function(x, eps=1e-6) pmin(pmax(x, eps), 1-eps)
rank_desc <- function(v){ r <- rep(NA_real_, length(v)); ok <- is.finite(v); r[ok] <- rank(-v[ok], ties.method="average"); r }
scale01 <- function(x){ r <- range(x, na.rm=TRUE); if (diff(r)==0) rep(0.5,length(x)) else (x-r[1])/diff(r) }
tmean_row <- function(v){
  vv <- v[is.finite(v)]
  if (length(vv) >= 3) { vv <- sort(vv); mean(vv[-c(1,length(vv))]) }
  else if (length(vv) > 0) mean(vv) else NA_real_
}

# Ranks por modelo (1=mejor)
R <- do.call(cbind, lapply(seq_len(M), function(j) rank_desc(P[,j])))

# tmean entre 7 (para fill y desempates)
TMEAN7 <- apply(P, 1, tmean_row)

# p_or entre 7 (fill alternativo)
POR7 <- 1 - apply(1 - clip01(P), 1, function(v){ prod(v[is.finite(v)]) })

# tmean/q75 FULL (fills globales)
tmean_full_path <- "ens_tmean_FULL_prediccion.txt"
q75_full_path   <- "ens_creative_q75_FULL_prediccion.txt"
TMEAN_ALL <- if (file.exists(tmean_full_path)) fread(tmean_full_path)[,prob] else NULL
Q75_ALL   <- if (file.exists(q75_full_path))   fread(q75_full_path)[,prob]   else NULL

# ---------- helpers de ranking/guardado ----------
make_scores_from_order <- function(ord_ids, all_ids) {
  ord_ids <- unique(ord_ids)
  full <- c(ord_ids, setdiff(all_ids, ord_ids))          # completa universo
  pos <- match(full, all_ids)
  sc <- numeric(length(all_ids))
  sc[pos] <- seq(from = 1, to = 0, length.out = length(all_ids))
  sc
}

save_pred <- function(name, sc){
  fn <- sprintf("ens_AF_%s_FULL_prediccion.txt", name)
  fwrite(data.table(numero_de_cliente = ALL$numero_de_cliente,
                    prob = scale01(sc)), fn, sep="\t")
  cat("✓", fn, "\n"); fn
}

# ---------- fábrica ANCHOR+FILL ----------
build_AF <- function(L, tau, score_fill, name){
  votes <- rowSums(R <= L, na.rm=TRUE)
  anchor_idx <- which(votes >= tau)
  cat(sprintf("[AF] L=%d tau=%d | anchor=%d\n", L, tau, length(anchor_idx)))

  rmean <- rowMeans(R, na.rm=TRUE)
  sf <- score_fill; sf[!is.finite(sf)] <- -Inf

  order_anchor <- anchor_idx[order(-sf[anchor_idx], rmean[anchor_idx])]
  rest_idx     <- setdiff(seq_len(N), order_anchor)
  order_rest   <- rest_idx[order(-sf[rest_idx], rmean[rest_idx])]
  ord_final_ids <- ALL$numero_de_cliente[c(order_anchor, order_rest)]

  sc <- make_scores_from_order(ord_final_ids, ALL$numero_de_cliente)
  save_pred(name, sc)
}

# ---- Generación de variantes ----
FILES_OUT <- c(
  build_AF(11000L, 5L, TMEAN7,        "L11000_T5_tmean"),
  build_AF(11000L, 6L, TMEAN7,        "L11000_T6_tmean"),
  build_AF(10900L, 5L, TMEAN7,        "L10900_T5_tmean"),
  build_AF(11100L, 5L, TMEAN7,        "L11100_T5_tmean"),
  if (!is.null(TMEAN_ALL)) build_AF(11000L, 5L, TMEAN_ALL, "L11000_T5_tmeanALL"),
  if (!is.null(Q75_ALL))   build_AF(11000L, 5L, Q75_ALL,   "L11000_T5_q75ALL"),
  build_AF(11000L, 5L, POR7,          "L11000_T5_or7")
)

# ================= SUBMITS robustos (toggle) =================
SUBMIT <- TRUE
VARIANTS_TO_SUBMIT <- c("L11000_T5_tmean","L11000_T6_tmean","L10900_T5_tmean",
                        "L11100_T5_tmean","L11000_T5_tmeanALL","L11000_T5_q75ALL")
K_SET <- c(10500L, 11000L)

COMPETITION   <- "data-mining-analista-sr-2025-a"
EXP_NAME_BASE <- "KA_AF_TOP7"
if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent=TRUE)
  if (!inherits(P,"try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION   <- P$kaggle$competencia
    if (!is.null(P$experimento))        EXP_NAME_BASE <- paste0(P$experimento, "_AF_TOP7")
  }
}

# --- helpers de submit seguro ---
kaggle_submit_safe <- function(csv, msg, comp, cooldown=65, retries=2) {
  cmd <- c("competitions","submit","-c", comp, "-f", csv, "-m", shQuote(msg))
  out <- tryCatch(system2("kaggle", cmd, stdout=TRUE, stderr=TRUE),
                  error = function(e) as.character(e))
  txt <- paste(out, collapse="\n")
  cat("Kaggle says:\n", txt, "\n")

  if (grepl("Too Many Requests|\\b429\\b", txt, ignore.case=TRUE)) {
    if (retries > 0) { Sys.sleep(cooldown); return(kaggle_submit_safe(csv, msg, comp, cooldown, retries-1)) }
    return(list(ok=FALSE, reason="ratelimit"))
  }
  if (grepl("submission limit|exceeded.*limit|daily.*limit", txt, ignore.case=TRUE)) {
    return(list(ok=FALSE, reason="limit"))
  }
  if (grepl("Successfully submitted", txt, fixed=TRUE)) return(list(ok=TRUE, reason="ok"))
  list(ok=FALSE, reason="other", raw=txt)
}

submit_many <- function(pred_names, out_prefix, comp, k_set) {
  dir.create("kaggle", showWarnings = FALSE)
  for (nm in pred_names) {
    pred <- sprintf("ens_AF_%s_FULL_prediccion.txt", nm)
    if (!file.exists(pred)) next
    ens <- fread(pred); setnames(ens, names(ens)[1:2], c("numero_de_cliente","prob"))
    setorder(ens, -prob, numero_de_cliente)
    for (k in k_set) {
      csv <- sprintf("kaggle/%s_%s_top_%d.csv", out_prefix, nm, k)
      sub <- ens[, .(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= k))]
      fwrite(sub, csv)
      msg <- sprintf("AF | %s | top=%d", nm, k)
      cat("Subiendo:", csv, "...\n")
      res <- kaggle_submit_safe(csv, msg, comp)
      if (identical(res$reason, "limit")) {
        message("Se detectó límite de envíos. Corto el loop de submits.")
        return(invisible(NULL))
      }
      Sys.sleep(8)  # anti-429
    }
  }
  invisible(NULL)
}

if (SUBMIT) {
  stopifnot(!inherits(try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE),"try-error"))
  submit_many(VARIANTS_TO_SUBMIT, EXP_NAME_BASE, COMPETITION, K_SET)
  cat("Listo. Si alguno supera 66.257, barrido fino de K alrededor del mejor.\n")
}



WF usados:
  • /home/juaniripoll27/buckets/b1/exp/WF9530/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9519/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9518/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9517/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9514/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9513/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9512/prediccion.txt 
Referencia: ens_tmean_FULL_prediccion.txt  | n= 165093 
[AF] L=11000 tau=5 | anchor=10197
✓ ens_AF_L11000_T5_tmean_FULL_prediccion.txt 
[AF] L=11000 tau=6 | anchor=8945
✓ ens_AF_L11000_T6_tmean_FULL_prediccion.txt 
[AF] L=10900 tau=5 | anchor=10094
✓ ens_AF_L10900_T5_tmean_FULL_prediccion.txt 
[AF] L=11100 tau=5 | anchor=10293
✓ ens_AF_L11100_T5_tmean_FULL_prediccion.txt 
[AF] L=11000 tau=5 | anchor=10197
✓ ens_AF_L11000_T5_tmeanALL_FULL_prediccion.txt 
[AF] L=11000 tau=5 | anchor=10197
✓ ens_AF_L11000_T5_q75ALL_FULL_prediccion.txt 
[AF] L=11000 tau=5 | anchor=

Warning message in system2("kaggle", cmd, stdout = TRUE, stderr = TRUE):
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA_AF_TOP7_L11000_T5_tmean_top_10500.csv -m 'AF | L11000_T5_tmean | top=10500' 2>&1' had status 1”


Kaggle says:
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.38MB/s]
400 Client Error: Bad Request for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Subiendo: kaggle/KA_AF_TOP7_L11000_T5_tmean_top_11000.csv ...


Warning message in system2("kaggle", cmd, stdout = TRUE, stderr = TRUE):
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA_AF_TOP7_L11000_T5_tmean_top_11000.csv -m 'AF | L11000_T5_tmean | top=11000' 2>&1' had status 1”


Kaggle says:
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.48MB/s]
400 Client Error: Bad Request for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Subiendo: kaggle/KA_AF_TOP7_L11000_T6_tmean_top_10500.csv ...


...

Kaggle says:
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.42MB/s]
429 Client Error: Too Many Requests for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 


Warning message in system2("kaggle", cmd, stdout = TRUE, stderr = TRUE):
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA_AF_TOP7_L11000_T5_q75ALL_top_11000.csv -m 'AF | L11000_T5_q75ALL | top=11000' 2>&1' had status 1”


Kaggle says:
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.60MB/s]
429 Client Error: Too Many Requests for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 


Warning message in system2("kaggle", cmd, stdout = TRUE, stderr = TRUE):
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA_AF_TOP7_L11000_T5_q75ALL_top_11000.csv -m 'AF | L11000_T5_q75ALL | top=11000' 2>&1' had status 1”


Kaggle says:
100%|██████████| 1.79M/1.79M [00:00<00:00, 3.60MB/s]
429 Client Error: Too Many Requests for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Listo. Si alguno supera 66.257, barrido fino de K alrededor del mejor.


In [33]:
# ======== LIST AGGREGATION: META-VOTES + INTERLEAVING + INTERSECT+FILL ========
if (!require(data.table)) install.packages("data.table"); library(data.table)
if (!require(yaml))       install.packages("yaml");       library(yaml)

WF_IDS    <- c(30,19,18,17,14,13,12)                 # 7 mejores (sin 9510)
WF_LABELS <- sprintf("WF95%02d", WF_IDS)

# --- localizar prediccion.txt ---
BASES <- unique(c(
  getwd(),
  Sys.getenv("EXP_BASE", unset = NA_character_),
  Sys.glob("/home/*/buckets/*/exp"),
  Sys.glob("/content/buckets/*/exp")
))
BASES <- BASES[!is.na(BASES) & dir.exists(BASES)]
stopifnot(length(BASES)>0)

find_pred <- function(wf){
  for (b in BASES) {
    p <- file.path(b, wf, "prediccion.txt")
    if (file.exists(p)) return(normalizePath(p))
  }
  NA_character_
}
FILES <- vapply(WF_LABELS, find_pred, FUN.VALUE=character(1))
if (anyNA(FILES)) stop("Faltan:\n  - ", paste(file.path(WF_LABELS[is.na(FILES)], "prediccion.txt"), collapse="\n  - "))
cat("WF usados:\n", paste(" •", FILES, collapse="\n"), "\n")

# --- leer predicciones ---
read_pred <- function(path, idx){
  dt <- fread(path)
  if (!all(c("numero_de_cliente","prob") %in% names(dt)))
    setnames(dt, names(dt)[1:2], c("numero_de_cliente","prob"))
  dt[, prob := as.numeric(prob)]
  dt <- dt[, .(prob = mean(prob, na.rm=TRUE)), by=numero_de_cliente]
  setnames(dt, "prob", paste0("p", idx))
  dt[]
}
DTs <- Map(read_pred, FILES, seq_along(FILES))

# --- referencia FULL (mismo universo Kaggle) ---
ref_candidates <- c(
  "ens_tmean_FULL_prediccion.txt",
  "ens_logit_WF_30_19_18_17_14_prediccion.txt",
  FILES[1]
)
ref_path <- ref_candidates[file.exists(ref_candidates)][1]
stopifnot(length(ref_path)>0)
REF <- fread(ref_path)[, .(numero_de_cliente)]
cat("Referencia:", ref_path, " | n=", nrow(REF), "\n")

# --- unión FULL de los 7 ---
UNION <- Reduce(function(a,b) merge(a,b,by="numero_de_cliente", all=TRUE), DTs)
ALL   <- merge(REF, UNION, by="numero_de_cliente", all.x=TRUE)
pred_cols <- grep("^p\\d+$", names(ALL), value=TRUE)
P <- as.matrix(ALL[, ..pred_cols]); N <- nrow(P); M <- ncol(P)

# utils
scale01 <- function(x){ r <- range(x, na.rm=TRUE); if (diff(r)==0) rep(0.5,length(x)) else (x-r[1])/diff(r) }
rank_desc <- function(v){ r <- rep(NA_real_, length(v)); ok <- is.finite(v); r[ok] <- rank(-v[ok], ties.method="average"); r }
tmean_row <- function(v){
  vv <- v[is.finite(v)]
  if (length(vv) >= 3) { vv <- sort(vv); mean(vv[-c(1,length(vv))]) }
  else if (length(vv) > 0) mean(vv) else NA_real_
}

# ranks por modelo (1=mejor)
R <- do.call(cbind, lapply(seq_len(M), function(j) rank_desc(P[,j])))

# --- ranking base A: votes (L=11000), desempate por tmean7 y rmean ---
L_BASE <- 11000L
votes  <- rowSums(R <= L_BASE, na.rm=TRUE)
TMEAN7 <- apply(P, 1, tmean_row)
rmean  <- rowMeans(R, na.rm=TRUE)
ord_votes <- order(-votes, -TMEAN7, rmean)                 # índices (posiciones 1..N)
LIST_V <- ALL$numero_de_cliente[ord_votes]                 # lista ordenada

# --- ranking base B: tmean_FULL (si no, tmean7) ---
tmean_full_path <- "ens_tmean_FULL_prediccion.txt"
TMEAN_ALL <- if (file.exists(tmean_full_path)) fread(tmean_full_path)[,prob] else TMEAN7
ord_tmean <- order(-TMEAN_ALL, rmean)
LIST_T <- ALL$numero_de_cliente[ord_tmean]

# --- ranking base C: q75_FULL si existe ---
q75_full_path <- "ens_creative_q75_FULL_prediccion.txt"
Q75_ALL <- if (file.exists(q75_full_path)) fread(q75_full_path)[,prob] else NULL
if (is.null(Q75_ALL)) message("Aviso: no hay q75_FULL; se omitirán variantes que lo requieren.")
if (!is.null(Q75_ALL)) {
  ord_q75 <- order(-Q75_ALL, rmean)
  LIST_Q  <- ALL$numero_de_cliente[ord_q75]
}

# helpers de armado
make_scores_from_order <- function(ord_ids){
  # genera 'prob' monotónica según orden; devuelve vector length N en orden de ALL
  pos <- match(ord_ids, ALL$numero_de_cliente)
  score_seq <- seq(from=1, to=0, length.out=N)
  sc <- numeric(N); sc[pos] <- score_seq
  sc
}
# interleaving ponderado entre 2 o 3 listas
interleave_unique <- function(lists, weights, K){
  stopifnot(length(lists)==length(weights))
  idx <- rep(1, length(lists))        # punteros
  chosen <- integer(0); chosen_set <- new.env(parent=emptyenv())
  push <- function(x){
    if (!exists(as.character(x), chosen_set, inherits=FALSE)){
      assign(as.character(x), TRUE, envir=chosen_set)
      chosen <<- c(chosen, x)
    }
  }
  # ciclos hasta juntar >=K
  while (length(chosen) < K && any(idx <= sapply(lists, length))) {
    for (j in seq_along(lists)) {
      take <- weights[j]
      while (take > 0 && idx[j] <= length(lists[[j]]) && length(chosen) < K) {
        push(lists[[j]][ idx[j] ])
        idx[j] <- idx[j] + 1
        take <- take - 1
      }
      if (length(chosen) >= K) break
    }
  }
  chosen
}

# intersección primero (common core), luego fill con interleave
intersect_then_fill <- function(listA, listB, a=9000, weights=c(8,2), K=10500){
  core <- intersect(listA[1:a], listB[1:a])
  # orden del core por suma de ranks relativos en A y B
  rA <- match(core, listA); rB <- match(core, listB)
  core_ord <- core[order(rA + rB)]
  # fill con interleave entre A y B evitando duplicados
  restA <- setdiff(listA, core_ord); restB <- setdiff(listB, core_ord)
  fill <- interleave_unique(list(restA, restB), weights, K - length(core_ord))
  c(core_ord, fill)
}

# =================== 1) META-VOTES (métodos) ===================
L_META <- 11000L
in_top <- list(
  V = LIST_V[1:L_META],
  T = LIST_T[1:L_META],
  Q = if (!is.null(Q75_ALL)) LIST_Q[1:L_META] else integer(0)
)
# conteo de presencia + desempate por suma de ranks en V,T,(Q)
rank_map <- function(lst){ m <- integer(N); m[match(lst, ALL$numero_de_cliente)] <- seq_along(lst); m[m==0] <- N+N }
RV <- rank_map(LIST_V); RT <- rank_map(LIST_T); RQ <- if (!is.null(Q75_ALL)) rank_map(LIST_Q) else rep(N+N, N)
present <- (RV <= L_META) + (RT <= L_META) + (RQ <= L_META)
ord_meta <- order(-present, RV + RT + RQ)
LIST_META <- ALL$numero_de_cliente[ord_meta]
SC_metavotes <- make_scores_from_order(LIST_META)

# =================== 2) INTERLEAVING 8:2 (V/T) ===================
LIST_8v2 <- interleave_unique(list(LIST_V, LIST_T), c(8,2), N)
SC_8v2 <- make_scores_from_order(LIST_8v2)

# =================== 3) INTERLEAVING 7:3 (V/T) ===================
LIST_7v3 <- interleave_unique(list(LIST_V, LIST_T), c(7,3), N)
SC_7v3 <- make_scores_from_order(LIST_7v3)

# =================== 4) INTERLEAVING 6:3:1 (V/T/Q) ===================
if (!is.null(Q75_ALL)) {
  LIST_6v3v1 <- interleave_unique(list(LIST_V, LIST_T, LIST_Q), c(6,3,1), N)
  SC_6v3v1 <- make_scores_from_order(LIST_6v3v1)
}

# =================== 5) INTERSECT 9000 (V∩T) + FILL 8:2 ===================
LIST_int9000 <- intersect_then_fill(LIST_V, LIST_T, a=9000, weights=c(8,2), K=N)
SC_int9000 <- make_scores_from_order(LIST_int9000)

# ---- Guardar prediccion.txt de todos ----
save_pred <- function(name, sc){
  fn <- sprintf("ens_listagg_%s_FULL_prediccion.txt", name)
  fwrite(data.table(numero_de_cliente = ALL$numero_de_cliente, prob = scale01(sc)), fn, sep="\t")
  cat("✓", fn, "\n"); fn
}
OUTS <- c(
  save_pred("metavotes_L11000", SC_metavotes),
  save_pred("interleave_8v2",    SC_8v2),
  save_pred("interleave_7v3",    SC_7v3),
  save_pred("intersect9000_8v2", SC_int9000),
  if (!is.null(Q75_ALL)) save_pred("interleave_6v3v1", SC_6v3v1)
)

# ===================== SUBMITS (toggle) =====================
SUBMIT <- TRUE
BLENDS_TO_SUBMIT <- c("metavotes_L11000","interleave_8v2","interleave_7v3","intersect9000_8v2","interleave_6v3v1")
K_SET <- c(10500L, 11000L, 11500L)

COMPETITION   <- "data-mining-analista-sr-2025-a"
EXP_NAME_BASE <- "KA_LISTAGG_TOP7"
if (file.exists("PARAM.yml")) {
  P <- try(read_yaml("PARAM.yml"), silent=TRUE)
  if (!inherits(P,"try-error")) {
    if (!is.null(P$kaggle$competencia)) COMPETITION   <- P$kaggle$competencia
    if (!is.null(P$experimento))        EXP_NAME_BASE <- paste0(P$experimento, "_LISTAGG_TOP7")
  }
}

if (SUBMIT) {
  stopifnot(!inherits(try(system2("kaggle","--version",stdout=TRUE,stderr=TRUE), silent=TRUE),"try-error"))
  dir.create("kaggle", showWarnings = FALSE)
  for (nm in BLENDS_TO_SUBMIT) {
    pred <- sprintf("ens_listagg_%s_FULL_prediccion.txt", nm)
    if (!file.exists(pred)) next
    ens <- fread(pred); setnames(ens, names(ens)[1:2], c("numero_de_cliente","prob"))
    setorder(ens, -prob, numero_de_cliente)
    for (k in K_SET) {
      out <- sprintf("kaggle/%s_%s_top_%d.csv", EXP_NAME_BASE, nm, k)
      sub <- ens[, .(numero_de_cliente, Predicted = as.integer(seq_len(.N) <= k))]
      fwrite(sub, out)
      msg <- sprintf("LISTAGG | %s | top=%d", nm, k)
      cat("Subiendo:", out, "...\n")
      res <- system2("kaggle",
        c("competitions","submit","-c", COMPETITION, "-f", out, "-m", shQuote(msg)),
        stdout=TRUE, stderr=TRUE)
      cat(paste(res, collapse="\n"), "\n")
      Sys.sleep(8)
    }
  }
  cat("Listo. Verificá si alguno supera 66.257. Si aparece un ganador, hacemos barrido fino de K (±300, paso 50).\n")
}


WF usados:
  • /home/juaniripoll27/buckets/b1/exp/WF9530/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9519/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9518/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9517/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9514/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9513/prediccion.txt
 • /home/juaniripoll27/buckets/b1/exp/WF9512/prediccion.txt 
Referencia: ens_tmean_FULL_prediccion.txt  | n= 165093 


Warning message in sc[pos] <- score_seq:
“number of items to replace is not a multiple of replacement length”


✓ ens_listagg_metavotes_L11000_FULL_prediccion.txt 
✓ ens_listagg_interleave_8v2_FULL_prediccion.txt 
✓ ens_listagg_interleave_7v3_FULL_prediccion.txt 
✓ ens_listagg_intersect9000_8v2_FULL_prediccion.txt 
✓ ens_listagg_interleave_6v3v1_FULL_prediccion.txt 
Subiendo: kaggle/KA_LISTAGG_TOP7_metavotes_L11000_top_10500.csv ...


Warning message in system2("kaggle", c("competitions", "submit", "-c", COMPETITION, :
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA_LISTAGG_TOP7_metavotes_L11000_top_10500.csv -m 'LISTAGG | metavotes_L11000 | top=10500' 2>&1' had status 1”


100%|██████████| 1.79M/1.79M [00:00<00:00, 3.84MB/s]
400 Client Error: Bad Request for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Subiendo: kaggle/KA_LISTAGG_TOP7_metavotes_L11000_top_11000.csv ...


...

100%|██████████| 1.79M/1.79M [00:00<00:00, 3.57MB/s]
400 Client Error: Bad Request for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Subiendo: kaggle/KA_LISTAGG_TOP7_interleave_6v3v1_top_11000.csv ...


Warning message in system2("kaggle", c("competitions", "submit", "-c", COMPETITION, :
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA_LISTAGG_TOP7_interleave_6v3v1_top_11000.csv -m 'LISTAGG | interleave_6v3v1 | top=11000' 2>&1' had status 1”


100%|██████████| 1.79M/1.79M [00:00<00:00, 3.57MB/s]
400 Client Error: Bad Request for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Subiendo: kaggle/KA_LISTAGG_TOP7_interleave_6v3v1_top_11500.csv ...


Warning message in system2("kaggle", c("competitions", "submit", "-c", COMPETITION, :
“running command ''kaggle' competitions submit -c data-mining-analista-sr-2025-a -f kaggle/KA_LISTAGG_TOP7_interleave_6v3v1_top_11500.csv -m 'LISTAGG | interleave_6v3v1 | top=11500' 2>&1' had status 1”


100%|██████████| 1.79M/1.79M [00:00<00:00, 3.46MB/s]
400 Client Error: Bad Request for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/data-mining-analista-sr-2025-a 
Listo. Verificá si alguno supera 66.257. Si aparece un ganador, hacemos barrido fino de K (±300, paso 50).


In [35]:
getwd()
normalizePath("kaggle", mustWork = FALSE)
list.files("kaggle", full.names = TRUE)  # acá vas a ver los .csv listos

# Copiarlos al bucket montado por gcsfuse:
dest <- "/home/juaniripoll27/buckets/b1/exp/kaggle"
dir.create(dest, showWarnings = FALSE, recursive = TRUE)
file.copy(list.files("kaggle", full.names = TRUE, pattern="\\.csv$"),
          dest, overwrite = TRUE)


[1] "/home/juaniripoll27/dm2025a/src/workflows"

[1] "/home/juaniripoll27/dm2025a/src/workflows/kaggle"

[1] "kaggle/KA_AF_TOP7_L10900_T5_tmean_top_10500.csv"                        
  [2] "kaggle/KA_AF_TOP7_L10900_T5_tmean_top_11000.csv"                        
  [3] "kaggle/KA_AF_TOP7_L11000_T5_q75ALL_top_10500.csv"                       
  [4] "kaggle/KA_AF_TOP7_L11000_T5_q75ALL_top_11000.csv"                       
  [5] "kaggle/KA_AF_TOP7_L11000_T5_tmean_top_10500.csv"                        
  [6] "kaggle/KA_AF_TOP7_L11000_T5_tmean_top_11000.csv"                        
  [7] "kaggle/KA_AF_TOP7_L11000_T5_tmeanALL_top_10500.csv"                     
  [8] "kaggle/KA_AF_TOP7_L11000_T5_tmeanALL_top_11000.csv"                     
  [9] "kaggle/KA_AF_TOP7_L11000_T6_tmean_top_10500.csv"                        
 [10] "kaggle/KA_AF_TOP7_L11000_T6_tmean_top_11000.csv"                        
 [11] "kaggle/KA_AF_TOP7_L11100_T5_tmean_top_10500.csv"                        
 [12] "kaggle/KA_AF_TOP7_L11100_T5_tmean_top_11000.csv"                        
 [13] "kaggle/KA_ALL_ENS_FULL_logit_div_top_10500.csv"                         
 [14] "kaggle/KA_ALL_ENS_FULL_logit_div_top_11000.csv"                         
 [15] "kaggle/KA_ALL_ENS_FULL_logit_div_top_11500.csv"                         
 [16] "kaggle/KA_ALL_ENS_logit_div_top_10500.csv"                              
 [17] "kaggle/KA_ALL_ENS_logit_div_top_11000.csv"                              
 [18] "kaggle/KA_ALL_ENS_logit_div_top_11500.csv"                              
 [19] "kaggle/KA_ALL_FULL_logit_div_top_10500.csv"                             
 [20] "kaggle/KA_ALL_FULL_logit_div_top_11000.csv"                             
 [21] "kaggle/KA_ALL_FULL_logit_div_top_11500.csv"                             
 [22] "kaggle/KA_ALL_FULL_logit_top_10500.csv"                                 
 [23] "kaggle/KA_ALL_FULL_logit_top_11000.csv"                                 
 [24] "kaggle/KA_ALL_FULL_logit_top_11500.csv"                                 
 [25] "kaggle/KA_ALL_FULL_mean_top_10500.csv"                                  
 [26] "kaggle/KA_ALL_FULL_mean_top_11000.csv"                                  
 [27] "kaggle/KA_ALL_FULL_mean_top_11500.csv"                                  
 [28] "kaggle/KA_ALL_FULL_median_top_10500.csv"                                
 [29] "kaggle/KA_ALL_FULL_median_top_11000.csv"                                
 [30] "kaggle/KA_ALL_FULL_median_top_11500.csv"                                
 [31] "kaggle/KA_ALL_FULL_rank_top_10500.csv"                                  
 [32] "kaggle/KA_ALL_FULL_rank_top_11000.csv"                                  
 [33] "kaggle/KA_ALL_FULL_rank_top_11500.csv"                                  
 [34] "kaggle/KA_ALL_FULL_tmean_top_10500.csv"                                 
 [35] "kaggle/KA_ALL_FULL_tmean_top_11000.csv"                                 
 [36] "kaggle/KA_ALL_FULL_tmean_top_11500.csv"                                 
 [37] "kaggle/KA_consensus07_tmean_wf30_19_18_17_14_13_12_L12000_top_10500.csv"
 [38] "kaggle/KA_consensus07_tmean_wf30_19_18_17_14_13_12_L12000_top_11000.csv"
 [39] "kaggle/KA_consensus07_tmean_wf30_19_18_17_14_13_12_L12000_top_11500.csv"
 [40] "kaggle/KA_CREATIVE_FULL_pmean3_top_10500.csv"                           
 [41] "kaggle/KA_CREATIVE_FULL_pmean3_top_11000.csv"                           
 [42] "kaggle/KA_CREATIVE_FULL_pmean3_top_11500.csv"                           
 [43] "kaggle/KA_CREATIVE_FULL_q75_top_10500.csv"                              
 [44] "kaggle/KA_CREATIVE_FULL_q75_top_11000.csv"                              
 [45] "kaggle/KA_CREATIVE_FULL_q75_top_11500.csv"                              
 [46] "kaggle/KA_CREATIVE_FULL_rrf_top_10500.csv"                              
 [47] "kaggle/KA_CREATIVE_FULL_rrf_top_11000.csv"                              
 [48] "kaggle/KA_CREATIVE_FULL_rrf_top_11500.csv"                              
 [49] "kaggle/KA_CREATIVE_FULL_smax_top_10500.csv"                             
 [50] "kaggle/KA_CREATIVE_FULL_smax_top_11000.csv"                             
 [

[1] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
 [16] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
 [31] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
 [46] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
 [61] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
 [76] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
 [91] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
[106] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE
[121] TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE TRUE